# Notebook 3 · Main model, re-run version: old Riyadh core vs Al-Waha (King Salman District)
**Analysis code for Al-Homoud, M. "Shaded but Not Cooler" (F1000Research).** The 1 m pedestrian heat model is the one validated in Al-Homoud (submitted), with new analyses.
Same workflow as Notebook 1, with longer automatic retries for the online data services; provided for re-running the analysis (no stored outputs).

| Part | What it measures | New in this notebook |
|---|---|---|
| A. Pedestrian heat (1 m, hourly) | Shade, mean radiant temperature (Tmrt), UTCI on every street's pedestrian space | Street widths **measured from the buildings**, so the narrow lanes of the old core are modelled correctly |
| B. Urban form | Local Climate Zones (Stewart & Oke) on a 100 m grid | New |
| C. Surface heat islands (30 m) | Landsat summer LST, **hot and cold spots** (Getis-Ord Gi*) in each district and a 1.5 km ring | New |
| D. City heat island, day and night (1 km) | MODIS Aqua LST at 13:30 and 01:30, **heat-island intensity against the surrounding desert**, year by year | New |
| E. What-if | **Cool pavements**, **shade sails**, and both | New (Al-Homoud, submitted, tested walls and trees) |
| F. Check | Official SOLWEIG model on 400 m windows, with matched sun timing | Matched sun timing |

**How to run:** **Runtime → Run all**. It takes about **1.5–2 hours**. All results are written straight into your **Google Drive** (folder `thermal_oldcore_alwaha`), so nothing is lost if the session disconnects: just run all again and the finished parts are reused.

**Nothing to upload.** The official boundaries (Geoportal exports of Al-Dirah, Al-Futah and Al-Waha) are already in Step 1.

## Step 0 · Install packages (≈2 min)

In [ ]:
!pip -q install overturemaps duckdb osmnx geopandas pyogrio rasterio pystac-client planetary-computer pythermalcomfort statsmodels folium

## Step 0 · Settings
You do not need to change anything. Allow Google Drive access when asked (it keeps your results safe).

In [ ]:
RES        = 1.0        # model resolution (m)
BUFFER_M   = 150        # margin so shadows from outside each district are included
UTC_OFFSET = 3
HOURS      = list(range(6, 20))      # 06:00–19:00
WEATHER_YEAR = 2025
DESIGN_DAY = 'auto'     # hottest clear-sky day of Jul–Aug WEATHER_YEAR (paper 1 used 2025-07-30)
FLOOR_H, DEFAULT_H = 3.3, 7.5
SW_W = 2.5              # sidewalk strip width (m)
SHARED_LANE_W = 8.0     # lanes narrower than this (facade to facade) are shared: the whole lane is pedestrian space
STREET_SOURCE = 'overture'   # same street source for both districts ('overture' is reliable; 'osm' uses the Overpass servers)

LST_YEARS, LST_MONTHS, LST_MAX_CLOUD = (2023, 2025), (6, 9), 10   # Landsat summer composite
CONTEXT_M   = 1500      # ring around each district for the 30 m hot-spot analysis
GI_RADIUS_M = 120       # Gi* neighbourhood radius
LCZ_CELL    = 100       # Local Climate Zone cell size (m)
MODIS_YEARS = (2013, 2025)           # MODIS Aqua summers for the day/night heat island
MODIS_MONTHS = (6, 9)
CITY_CENTRE = (24.6877, 46.7219)     # Riyadh (lat, lon) for the 1 km heat-island map
RURAL_RING_KM = (40, 70)             # rural desert reference ring around the centre

RUN_SCENARIOS = True
COOL_ALBEDO = 0.45      # cool (solar-reflective) pavement albedo on streets and pedestrian space
SAIL_BASE, SAIL_TOP = 4.0, 4.3       # shade-sail height (m); sails let ~3% of the sun through
OVERTURE_RELEASE = '2026-10-21.0'    # used only if the overturemaps tool fails

USE_DRIVE = True
import os
OUT = '/content/thermal_oldcore_alwaha'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT = '/content/drive/MyDrive/thermal_oldcore_alwaha'
    except Exception as e:
        print('Google Drive not mounted, saving in the session only:', str(e)[:80])
os.makedirs(OUT, exist_ok=True)
print('Results folder:', OUT)

## Step 0 · The model code
These cells write the model to files. Do not change them.
- `tw_core.py`: sun position, shadows from buildings and trees, sky view factor, mean radiant temperature (SOLWEIG-type), exactly as validated in paper 1.
- `tw_analysis.py`: the hour-by-hour day simulation.
- `tw_extra.py`: new: measured street widths, Local Climate Zones, Gi* hot spots.

In [ ]:
%%writefile tw_core.py
# ---- Thermal-walkability core engine (pure numpy) ----
# Sun position (NOAA), shadow casting for buildings + trees (Ratti & Richens 1990 /
# Lindberg et al. 2008 style), sky view factor from horizon angles, and a SOLWEIG-type
# simplified radiation scheme for mean radiant temperature (Tmrt).
import numpy as np

SIGMA = 5.67e-8


def sun_position(dt_local, lat, lon, utc_offset):
    """Solar altitude and azimuth (degrees, azimuth clockwise from north). NOAA algorithm."""
    doy = dt_local.timetuple().tm_yday
    hour = dt_local.hour + dt_local.minute / 60
    g = 2 * np.pi / 365 * (doy - 1 + (hour - 12) / 24)
    eqt = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                    - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = hour * 60 + eqt + 4 * lon - 60 * utc_offset
    ha = np.radians(tst / 4 - 180)
    la = np.radians(lat)
    cz = np.sin(la) * np.sin(decl) + np.cos(la) * np.cos(decl) * np.cos(ha)
    zen = np.arccos(np.clip(cz, -1, 1))
    alt = 90 - np.degrees(zen)
    caz = (np.sin(la) * np.cos(zen) - np.sin(decl)) / (np.cos(la) * np.sin(zen))
    az = np.degrees(np.arccos(np.clip(caz, -1, 1)))
    az = (az + 180) % 360 if ha > 0 else (540 - az) % 360
    return float(alt), float(az)


def _shift(a, dy, dx, fill):
    """out[r, c] = a[r + dy, c + dx] (no wrap-around)."""
    out = np.full_like(a, fill)
    H, W = a.shape
    r0, r1 = max(0, -dy), min(H, H - dy)
    c0, c1 = max(0, -dx), min(W, W - dx)
    if r1 > r0 and c1 > c0:
        out[r0:r1, c0:c1] = a[r0 + dy:r1 + dy, c0 + dx:c1 + dx]
    return out


def shadows(dsm, alt, az, res=1.0, veg_top=None, veg_base=None, veg_trans=0.03, max_dist=300.0):
    """Return (sun_bldg, sun_all): 1 = sunlit, 0 = shaded. Rows grow southwards (north-up raster).
    dsm: absolute surface height (ground + buildings). veg_top/veg_base: absolute canopy top / trunk base
    (np.nan where no tree). Tree shade lets veg_trans of the beam through."""
    if alt <= 0.5:
        z = np.zeros_like(dsm)
        return z, z
    tan_a = np.tan(np.radians(alt))
    azr = np.radians(az)
    sx, sy = np.sin(azr), -np.cos(azr)          # step towards the sun in (col, row)
    span = np.nanmax(dsm) - np.nanmin(dsm)
    if veg_top is not None and np.isfinite(veg_top).any():
        span = max(span, np.nanmax(veg_top) - np.nanmin(dsm))
    dmax = min(max_dist, span / tan_a + res)
    nsteps = int(np.ceil(dmax / res * max(abs(sx), abs(sy)))) + 1
    blocked = np.zeros(dsm.shape, bool)
    # own pixel: the ray enters the crown above the pedestrian when the crown base is lower than the
    # height the ray reaches within half a pixel (high sun), otherwise neighbouring crown pixels decide
    if veg_top is not None:
        vblock = (np.nan_to_num(veg_top, nan=-1e9) > dsm) & (np.nan_to_num(veg_base, nan=1e9) - dsm < 0.5 * res * tan_a)
    else:
        vblock = np.zeros(dsm.shape, bool)
    last = (0, 0)
    for i in range(1, nsteps + 1):
        f = i / max(abs(sx), abs(sy))           # step length in pixels along the ray
        dx, dy = int(round(f * sx)), int(round(f * sy))
        if (dx, dy) == last:
            continue
        last = (dx, dy)
        d = np.hypot(dx, dy) * res
        ray = dsm + d * tan_a
        blocked |= _shift(dsm, dy, dx, -1e9) > ray
        if veg_top is not None:
            vt = _shift(veg_top, dy, dx, np.nan)
            vb = _shift(veg_base, dy, dx, np.nan)
            vblock |= (vt > ray) & (vb < ray)
        if d > dmax:
            break
    sun_b = (~blocked).astype('float32')
    sun_all = sun_b * np.where(vblock, veg_trans, 1.0).astype('float32')
    return sun_b, sun_all


def sky_view_factor(dsm, res=1.0, veg_top=None, n_az=24, max_dist=100.0, veg_trans=0.03):
    """SVF from horizon angles on n_az azimuths (isotropic sky, SVF = 1 - mean sin^2(horizon)).
    Returns (svf_bldg, svf_all)."""
    dists = np.unique(np.round(np.concatenate([np.arange(1, 20, 1), np.arange(20, 50, 2),
                                               np.arange(50, max_dist + 1, 5)]) / res)).astype(int)
    acc_b = np.zeros(dsm.shape, 'float32')                 # running sum of sin^2(horizon)
    acc_v = np.zeros(dsm.shape, 'float32')
    for k in range(n_az):
        a = 2 * np.pi * k / n_az
        sx, sy = np.sin(a), -np.cos(a)
        best_b = np.zeros(dsm.shape, 'float32')
        best_v = np.zeros(dsm.shape, 'float32') if veg_top is not None else None
        for p in dists:
            dx, dy = int(round(p * sx)), int(round(p * sy))
            d = np.hypot(dx, dy) * res
            if d == 0:
                continue
            tb = (_shift(dsm, dy, dx, -1e9) - dsm) / d
            np.maximum(best_b, tb, out=best_b)
            if veg_top is not None:
                tv = (_shift(veg_top, dy, dx, np.nan) - dsm) / d
                np.fmax(best_v, tv, out=best_v)
        acc_b += np.sin(np.arctan(np.clip(best_b, 0, None))) ** 2
        if veg_top is not None:
            acc_v += np.sin(np.arctan(np.clip(np.fmax(best_b, best_v), 0, None))) ** 2
    svf_b = 1 - acc_b / n_az
    if veg_top is None:
        return svf_b.astype('float32'), svf_b.astype('float32')
    svf_bv = 1 - acc_v / n_az
    svf_all = svf_b - (svf_b - svf_bv) * (1 - veg_trans)
    return svf_b.astype('float32'), svf_all.astype('float32')


def sky_emissivity(ta, rh):
    """Clear-sky emissivity, Prata (1996), as used in SOLWEIG."""
    es = 6.112 * np.exp(17.62 * ta / (243.12 + ta))
    ea = rh / 100 * es
    w = 46.5 * ea / (ta + 273.15)
    return 1 - (1 + w) * np.exp(-np.sqrt(1.2 + 3 * w))


def surface_temp_excess(hour, sunrise, altmax, alt, ghi, TgK=0.37, Tstart=-3.41, TmaxLST=15.0):
    """Sunlit ground/wall temperature above air temperature (K), SOLWEIG parameterisation
    (Lindberg et al. 2008, 2016; UMEP default for paved surfaces): amplitude = TgK*altmax + Tstart,
    sinusoidal from sunrise to TmaxLST, scaled by a clearness index."""
    if alt <= 0 or hour <= sunrise:
        return 0.0
    amp = max(TgK * altmax + Tstart, 0.0)
    tg = amp * np.sin((hour - sunrise) / (TmaxLST - sunrise) * np.pi / 2)
    corr = 0.1473 * np.log(max(alt, 0.5)) + 0.3454
    g0 = 1098 * np.sin(np.radians(alt)) * np.exp(-0.057 / np.sin(np.radians(alt)))
    ci = min(1.0, ghi / max(g0, 1.0) + (1 - corr))
    return float(max(tg, 0.0) * max(ci, 0.0))


def tmrt_simplified(ta, rh, ghi, dni, dhi, alt, az, sun, svf, tg_add, tw_add, svf_bldg=None, albedo_g=0.20,
                    albedo_w=0.20, eps_g=0.95, eps_w=0.90, abs_k=0.70, eps_p=0.97):
    """SOLWEIG-type (Lindberg et al. 2008) six-direction radiation budget for a standing person.
    sun: 0..1 beam transmission; svf: total sky view factor (buildings + trees); svf_bldg: sky view factor
    of buildings only. The view blocked by buildings is hot wall (ta + tw_add), the view blocked by trees
    is foliage at air temperature, as in SOLWEIG. tg_add / tw_add from surface_temp_excess(). Returns Tmrt (C)."""
    if svf_bldg is None:
        svf_bldg = svf
    s = np.radians(max(alt, 0))
    beam_h = dni * np.sin(s) * sun                          # direct on horizontal
    walls = 1 - svf_bldg                                    # view to building walls
    veg = np.clip(svf_bldg - svf, 0, 1)                     # view to tree foliage
    k_wall = ghi * albedo_w * walls * 0.5                   # reflected by walls
    k_down = beam_h + dhi * svf + k_wall
    k_up = albedo_g * (beam_h + dhi * svf)
    k_lat = 0.5 * dhi * svf + 0.5 * k_up + k_wall
    azr = np.radians(az)
    k_dir_side = 0.22 * dni * np.cos(s) * (abs(np.sin(azr)) + abs(np.cos(azr))) * sun
    tg = ta + tg_add * sun                                  # sunlit ground heats, shaded ground ~ air
    tw = ta + tw_add
    l_sky = sky_emissivity(ta, rh) * SIGMA * (ta + 273.15) ** 4
    l_wall = eps_w * SIGMA * (tw + 273.15) ** 4
    l_veg = eps_w * SIGMA * (ta + 273.15) ** 4
    l_down = svf * l_sky + walls * l_wall + veg * l_veg
    l_up = eps_g * SIGMA * (tg + 273.15) ** 4
    l_lat = 0.5 * l_down + 0.5 * l_up
    s_str = (abs_k * (0.06 * k_down + 0.06 * k_up + 0.88 * k_lat + k_dir_side)
             + eps_p * (0.06 * l_down + 0.06 * l_up + 0.88 * l_lat))
    return ((s_str / (eps_p * SIGMA)) ** 0.25 - 273.15).astype('float32')


def day_geometry(day, lat, lon, utc_offset):
    """Sunrise (decimal hour) and maximum solar altitude of the day."""
    from datetime import datetime
    mins = np.arange(180, 1260, 2)
    alts = np.array([sun_position(datetime(day.year, day.month, day.day, int(m // 60), int(m % 60)),
                                  lat, lon, utc_offset)[0] for m in mins])
    return float(mins[np.argmax(alts > 0)] / 60), float(alts.max())


# ---- scenario helpers (raster-based) ----
def frontage_walls(sidewalk, carriage, ground):
    """Plot-boundary walls: ground pixels touching the outer edge of the sidewalk strip
    (not sidewalk, not carriageway, not building). Returns bool mask."""
    from scipy.ndimage import binary_dilation
    ring = binary_dilation(sidewalk, iterations=1) & ~sidewalk & ~carriage & ground
    return ring


def street_trees(sidewalk, carriage, ground, res=1.0, spacing=8.0, crown_r=2.5):
    """Street trees in the sidewalk next to the kerb, about one per `spacing` m.
    Returns bool mask of crown pixels (crowns may overhang the road, never buildings)."""
    from scipy.ndimage import binary_dilation, distance_transform_edt
    kerb = binary_dilation(carriage, iterations=max(1, int(round(1.0 / res)))) & sidewalk
    rr, cc = np.nonzero(kerb)
    if rr.size == 0:
        return np.zeros(sidewalk.shape, bool)
    cell = max(1, int(round(spacing / res)))
    key = (rr // cell) * (sidewalk.shape[1] // cell + 1) + (cc // cell)
    _, first = np.unique(key, return_index=True)
    pts = np.zeros(sidewalk.shape, bool); pts[rr[first], cc[first]] = True
    crown = distance_transform_edt(~pts) * res <= crown_r
    return crown & ground


def zonal_mean(labels, values, n):
    """Mean of values per label 1..n (label 0 = outside). Returns array length n+1 (index = label)."""
    ok = (labels > 0) & np.isfinite(values)
    s = np.bincount(labels[ok], weights=values[ok], minlength=n + 1)
    c = np.bincount(labels[ok], minlength=n + 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        return s / c

In [ ]:
%%writefile tw_analysis.py
# ---- Thermal-walkability day simulation and segment statistics ----
import numpy as np
import pandas as pd
from datetime import datetime
from tw_core import sun_position, shadows, tmrt_simplified, zonal_mean, surface_temp_excess, day_geometry


def run_day(day, wx, dsm, veg_top, veg_base, ground, svf, labels, n_seg, lat, lon,
            utc_offset=3, hours=range(6, 20), res=1.0, utci_fn=None, albedo_g=None,
            official=None, key_hours=(9, 12, 15, 18), log=print, tg_scale=None):
    """Simulate one day hour by hour.
    wx: DataFrame indexed by local datetime with ta, rh, ws, ghi, dni, dhi.
    ground: bool array (True = open ground, not roof). svf: (svf_bldg, svf_all).
    labels: int array, sidewalk pixels labelled 1..n_seg (0 = not sidewalk).
    official: optional dict hour -> (tmrt_array, utci_array or None) from the official SOLWEIG run.
    tg_scale: optional array; multiplies the sunlit-ground temperature excess (cool-pavement scenario: (1-a_new)/(1-a_old)).
    Returns seg_rows (DataFrame), dist_rows (DataFrame), daily rasters (dict), key-hour rasters (dict)."""
    svf_b, svf_all = svf
    sw = labels > 0
    sun_hours = np.zeros(dsm.shape, 'float32')
    h38 = np.zeros(dsm.shape, 'float32')
    h46 = np.zeros(dsm.shape, 'float32')
    tmrt_sum = np.zeros(dsm.shape, 'float32')
    key = {}
    seg_rows, dist_rows = [], []
    alb = 0.20 if albedo_g is None else albedo_g
    sunrise, altmax = day_geometry(day, lat, lon, utc_offset)
    for h in hours:
        t = datetime(day.year, day.month, day.day, h)
        if t not in wx.index:
            log(f'  {t}: no weather, skipped'); continue
        w = wx.loc[t]
        alt, az = sun_position(t, lat, lon, utc_offset)
        sb, sa = shadows(dsm, alt, az, res, veg_top, veg_base)
        shade_b, shade_all = 1 - sb, 1 - sa
        src = 'builtin'
        if official and h in official:
            tmrt, utci = official[h]; src = 'solweig'
        else:
            tg = surface_temp_excess(h, sunrise, altmax, alt, w.ghi)
            tga = tg if tg_scale is None else tg * tg_scale
            tmrt = tmrt_simplified(w.ta, w.rh, w.ghi, w.dni, w.dhi, alt, az, sa, svf_all, tga, tg, svf_bldg=svf_b, albedo_g=alb)
            utci = None
        tmrt = np.where(ground, tmrt, np.nan).astype('float32')
        if utci is None:
            utci = utci_fn(w.ta, tmrt, w.ws, w.rh) if utci_fn else np.full_like(tmrt, np.nan)
        utci = np.where(ground, utci, np.nan).astype('float32')
        sun_hours += np.where(ground, sa, 0)
        h38 += np.nan_to_num(utci >= 38)
        h46 += np.nan_to_num(utci >= 46)
        tmrt_sum += np.nan_to_num(tmrt)
        m_all = zonal_mean(labels, shade_all, n_seg)
        m_b = zonal_mean(labels, shade_b, n_seg)
        m_t = zonal_mean(labels, tmrt, n_seg)
        m_u = zonal_mean(labels, utci, n_seg)
        seg_rows.append(pd.DataFrame({'seg': np.arange(1, n_seg + 1), 'hour': h,
                                      'shade_pct': 100 * m_all[1:], 'shade_bldg_pct': 100 * m_b[1:],
                                      'tmrt': m_t[1:], 'utci': m_u[1:]}))
        dist_rows.append({'hour': h, 'sun_alt': round(alt, 1), 'sun_az': round(az, 1), 'ta': w.ta, 'rh': w.rh,
                          'ws': w.ws, 'ghi': w.ghi, 'engine': src,
                          'sidewalk_shade_pct': 100 * float(np.mean(shade_all[sw])),
                          'sidewalk_shade_bldg_pct': 100 * float(np.mean(shade_b[sw])),
                          'sidewalk_tmrt': float(np.nanmean(tmrt[sw])),
                          'sidewalk_utci': float(np.nanmean(utci[sw])),
                          'sidewalk_pct_utci_ge38': 100 * float(np.nanmean(utci[sw] >= 38)),
                          'sidewalk_pct_utci_ge46': 100 * float(np.nanmean(utci[sw] >= 46))})
        if h in key_hours:
            key[h] = {'shade': shade_all.astype('float32'), 'tmrt': tmrt, 'utci': utci}
        log(f'  {h:02d}:00  sun {alt:5.1f}°  sidewalk shade {dist_rows[-1]["sidewalk_shade_pct"]:5.1f}%  '
            f'Tmrt {dist_rows[-1]["sidewalk_tmrt"]:5.1f}  UTCI {dist_rows[-1]["sidewalk_utci"]:5.1f}  [{src}]')
    n_h = max(len(dist_rows), 1)
    seg = pd.concat(seg_rows, ignore_index=True)
    seg['shade_tree_pct'] = (seg.shade_pct - seg.shade_bldg_pct).clip(lower=0)
    daily = {'sun_hours': sun_hours, 'utci_ge38_hours': h38, 'utci_ge46_hours': h46,
             'tmrt_mean': tmrt_sum / n_h}
    return seg, pd.DataFrame(dist_rows), daily, key


def segment_summary(seg_hourly, daily, labels, n_seg, hours_day=range(7, 19)):
    """One row per segment: shade at key hours, daytime means, heat-stress hours."""
    p = seg_hourly.pivot(index='seg', columns='hour')
    out = pd.DataFrame(index=np.arange(1, n_seg + 1))
    for h in (9, 12, 15, 18):
        if ('shade_pct', h) in p:
            out[f'shade_{h:02d}'] = p[('shade_pct', h)]
            out[f'utci_{h:02d}'] = p[('utci', h)]
            out[f'tmrt_{h:02d}'] = p[('tmrt', h)]
    day = seg_hourly[seg_hourly.hour.isin(list(hours_day))].groupby('seg')
    out['shade_day_mean'] = day.shade_pct.mean()
    out['shade_bldg_day_mean'] = day.shade_bldg_pct.mean()
    out['shade_tree_day_mean'] = day.shade_tree_pct.mean()
    out['utci_day_mean'] = day.utci.mean()
    aft = seg_hourly[seg_hourly.hour.isin([15, 16, 17, 18])].groupby('seg')
    out['shade_afternoon'] = aft.shade_pct.mean()
    for k in ('utci_ge38_hours', 'utci_ge46_hours', 'sun_hours'):
        out[k] = zonal_mean(labels, daily[k], n_seg)[1:]
    out.index.name = 'seg'
    return out

In [ ]:
%%writefile tw_extra.py
# ---- Extra analyses for the old-core vs Al-Waha study ----
# 1. Pedestrian realm from MEASURED street widths (works for narrow traditional lanes and wide villa streets)
# 2. Local Climate Zones (Stewart & Oke 2012) from the 1 m morphology, on a 100 m grid
# 3. Getis-Ord Gi* hot/cold spots on a raster, with Benjamini-Hochberg FDR
import math
import numpy as np
import pandas as pd


# ------------------------------------------------------------------ 1. pedestrian realm
def measured_half_width(e, free_dist, transform, res, step=2.0):
    """Median distance (m) from each segment centreline to the nearest building facade.
    free_dist: distance-to-nearest-building raster (m)."""
    T = transform
    H, W = free_dist.shape
    out = []
    for gm in e.geometry:
        n = max(2, int(gm.length // step))
        pts = [gm.interpolate(i / (n - 1), normalized=True) for i in range(n)]
        vals = []
        for p in pts:
            c, r = int((p.x - T.c) / T.a), int((p.y - T.f) / T.e)
            if 0 <= r < H and 0 <= c < W:
                vals.append(free_dist[r, c])
        v = np.array(vals, 'float32')
        v = v[v > 0]
        out.append(float(np.median(v)) if v.size else np.nan)
    return np.array(out)


def realm_geometry(gm, cls, half_class, hw_meas, sw_w=2.5, shared_max_w=8.0, plaza_half=4.0):
    """Return (pedestrian_polygon, carriageway_polygon_or_None, street_width_m, realm_type).
    Rules (applied identically to both districts):
      - pedestrian-only ways (footway/pedestrian/path/steps) -> a band of min(measured half-width, 4 m) each side;
      - lanes whose facade-to-facade width < shared_max_w (8 m) -> shared lane: the whole lane is pedestrian realm;
      - streets with room for a sidewalk -> 2.5 m sidewalk strips outside the carriageway, where the carriageway
        half-width is the class value, reduced when the facades are closer than class value + sidewalk."""
    if not np.isfinite(hw_meas) or hw_meas <= 0:
        hw_meas = half_class + sw_w                 # no buildings nearby: fall back to the class table
    W = 2 * hw_meas
    if half_class == 0:
        hw = min(hw_meas, plaza_half)
        return gm.buffer(max(hw, 1.0), cap_style=2), None, W, 'pedestrian way'
    if W < shared_max_w:
        return gm.buffer(max(hw_meas, 1.0), cap_style=2), None, W, 'shared lane'
    carr = min(half_class, max(hw_meas - sw_w, 1.5))
    ped = gm.buffer(carr + sw_w, cap_style=2).difference(gm.buffer(carr, cap_style=2))
    return ped, gm.buffer(carr, cap_style=2), W, 'street with sidewalks'


# ------------------------------------------------------------------ 2. Local Climate Zones
# Stewart & Oke (2012), Table 3: SVF, aspect ratio, building surface fraction (%), impervious surface fraction (%),
# pervious surface fraction (%), height of roughness elements (m). Only zones that can occur here are listed.
LCZ_TABLE = {
    'LCZ 1 compact high-rise': dict(svf=(0.2, 0.4), ar=(2, 6), bsf=(40, 60), isf=(40, 60), psf=(0, 10), h=(25, 80)),
    'LCZ 2 compact mid-rise':  dict(svf=(0.3, 0.6), ar=(0.75, 2), bsf=(40, 70), isf=(30, 50), psf=(0, 20), h=(10, 25)),
    'LCZ 3 compact low-rise':  dict(svf=(0.2, 0.6), ar=(0.75, 1.5), bsf=(40, 70), isf=(20, 50), psf=(0, 30), h=(3, 10)),
    'LCZ 4 open high-rise':    dict(svf=(0.5, 0.7), ar=(0.75, 1.25), bsf=(20, 40), isf=(30, 40), psf=(30, 40), h=(25, 80)),
    'LCZ 5 open mid-rise':     dict(svf=(0.5, 0.8), ar=(0.3, 0.75), bsf=(20, 40), isf=(30, 50), psf=(20, 40), h=(10, 25)),
    'LCZ 6 open low-rise':     dict(svf=(0.6, 0.9), ar=(0.3, 0.75), bsf=(20, 40), isf=(20, 50), psf=(30, 60), h=(3, 10)),
    'LCZ 8 large low-rise':    dict(svf=(0.7, 1.0), ar=(0.1, 0.3), bsf=(30, 50), isf=(40, 50), psf=(0, 20), h=(3, 10)),
    'LCZ 9 sparsely built':    dict(svf=(0.8, 1.0), ar=(0.1, 0.25), bsf=(10, 20), isf=(0, 20), psf=(60, 80), h=(3, 10)),
    'LCZ E bare rock or paved': dict(svf=(0.9, 1.0), ar=(0, 0.1), bsf=(0, 10), isf=(90, 100), psf=(0, 10), h=(0, 0.25)),
    'LCZ F bare soil or sand': dict(svf=(0.9, 1.0), ar=(0, 0.1), bsf=(0, 10), isf=(0, 10), psf=(90, 100), h=(0, 0.25)),
}
# Scales put the weight on 3D form (SVF, aspect ratio, building fraction, height): in desert cities paved and bare
# ground are hard to separate from open data, so surface-cover fractions get half the weight.
_SCALE = dict(svf=0.15, ar=0.3, bsf=12, isf=40, psf=40, h=5)


def lcz_classify(props, footprint=None, large_m2=1000.0):
    """props: dict of arrays (svf, ar, bsf, isf, psf, h). Nearest Stewart & Oke class by summed range distance.
    footprint: mean building footprint (m2) per cell. LCZ 8 is by definition large low-rise buildings
    (warehouses, malls), so it is only allowed where the mean footprint exceeds large_m2."""
    names = list(LCZ_TABLE)
    cost = []
    for n in names:
        c = 0
        for k, (lo, hi) in LCZ_TABLE[n].items():
            x = props[k]
            c = c + (np.clip(lo - x, 0, None) + np.clip(x - hi, 0, None)) / _SCALE[k]
        if n.startswith('LCZ 8') and footprint is not None:
            c = np.where(footprint >= large_m2, c, np.inf)
        cost.append(c)
    cost = np.stack(cost)
    idx = np.nanargmin(np.where(np.isfinite(cost), cost, np.inf), axis=0)
    return np.array(names, dtype=object)[idx], np.nanmin(cost, axis=0)


def morphology_cells(bh, can, svf, impervious, ground, grid, cell=100.0, fp_px=None):
    """Aggregate 1 m morphology to `cell` m cells. impervious: bool raster of paved ground (not buildings).
    fp_px: raster holding, at each building pixel, the footprint area (m2) of that building (from the polygons).
    Returns DataFrame with one row per cell (row, col, x, y, bsf, isf, psf, h, svf, ar, canopy) + LCZ."""
    T, W, H = grid
    res = T.a
    f = int(round(cell / res))
    h2, w2 = (H // f) * f, (W // f) * f
    def agg(a, how='mean'):
        a = a[:h2, :w2].reshape(h2 // f, f, w2 // f, f)
        return np.nanmean(a, axis=(1, 3)) if how == 'mean' else np.nansum(a, axis=(1, 3))
    b = (bh > 0).astype('float32')
    bsf = 100 * agg(b)
    hb = np.where(bh > 0, bh, np.nan)
    with np.errstate(invalid='ignore'):
        h = agg(np.nan_to_num(hb) * b, 'sum') / np.maximum(agg(b, 'sum'), 1)
    isf = 100 * agg((impervious & ground).astype('float32'))
    psf = np.clip(100 - bsf - isf, 0, 100)
    sv = agg(np.where(ground, svf, np.nan))
    canopy = 100 * agg((can > 0).astype('float32'))
    # aspect ratio H/W from the canyon relation SVF = cos(atan(2 H/W)) (Oke 1987), using the cell's mean ground SVF
    ar = np.clip(np.tan(np.arccos(np.clip(sv, 0.05, 1))) / 2, 0, 6)
    if fp_px is None:                                                   # fallback: connected building patches
        from scipy.ndimage import label, binary_erosion
        lab, _ = label(binary_erosion(bh > 0))
        size = np.bincount(lab.ravel()) * res * res
        fp_px = np.where(lab > 0, size[lab], np.nan).astype('float32')
    fp_px = np.where(bh > 0, fp_px, np.nan)
    footprint = agg(fp_px)                                              # area-weighted mean footprint per cell
    lcz, cost = lcz_classify(dict(svf=sv, ar=ar, bsf=bsf, isf=isf, psf=psf, h=h), footprint)
    rr, cc = np.mgrid[0:h2 // f, 0:w2 // f]
    x = T.c + (cc + 0.5) * cell
    y = T.f - (rr + 0.5) * cell
    return pd.DataFrame({'row': rr.ravel(), 'col': cc.ravel(), 'x': x.ravel(), 'y': y.ravel(), 'bsf': bsf.ravel(),
                         'isf': isf.ravel(), 'psf': psf.ravel(), 'h': h.ravel(), 'svf': sv.ravel(), 'ar': ar.ravel(),
                         'canopy_pct': canopy.ravel(), 'footprint_m2': footprint.ravel(), 'lcz': lcz.ravel(),
                         'lcz_fit_cost': cost.ravel()})


# ------------------------------------------------------------------ 3. Getis-Ord Gi* on a raster
def getis_ord_star(a, radius_px):
    """Gi* z-scores with binary weights inside a circle of radius_px (self included). NaN cells are ignored."""
    from scipy.ndimage import convolve
    from scipy.stats import norm
    ok = np.isfinite(a)
    x = np.where(ok, a, 0.0).astype('float64')
    n = ok.sum()
    xbar = x[ok].mean()
    s = np.sqrt((x[ok] ** 2).mean() - xbar ** 2)
    r = int(radius_px)
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    k = ((xx ** 2 + yy ** 2) <= r * r).astype('float64')
    sw = convolve(ok.astype('float64'), k, mode='constant')          # sum of weights (valid neighbours)
    swx = convolve(x, k, mode='constant')
    with np.errstate(invalid='ignore', divide='ignore'):
        z = (swx - xbar * sw) / (s * np.sqrt((n * sw - sw ** 2) / (n - 1)))
    z[~ok] = np.nan
    p = 2 * norm.sf(np.abs(np.nan_to_num(z)))
    p[~ok] = np.nan
    return z.astype('float32'), p.astype('float32')


def bh_fdr(p, alpha=0.05):
    """Benjamini-Hochberg: boolean array, True where significant at FDR alpha."""
    pv = p[np.isfinite(p)]
    if pv.size == 0:
        return np.zeros(p.shape, bool)
    srt = np.sort(pv)
    m = srt.size
    below = srt <= alpha * np.arange(1, m + 1) / m
    thr = srt[np.nonzero(below)[0].max()] if below.any() else -1
    return np.isfinite(p) & (p <= thr)

In [ ]:
import sys, json, glob, math, zipfile, subprocess, warnings, requests, time
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import shapely, rasterio
from shapely.geometry import box, Polygon
from shapely.ops import unary_union
from rasterio.features import rasterize
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from scipy.ndimage import distance_transform_edt
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
import importlib, tw_core, tw_analysis, tw_extra
for m in (tw_core, tw_analysis, tw_extra): importlib.reload(m)
from tw_core import sun_position, shadows, sky_view_factor, tmrt_simplified, zonal_mean, surface_temp_excess, day_geometry
from tw_analysis import run_day, segment_summary
from tw_extra import measured_half_width, realm_geometry, morphology_cells, getis_ord_star, bh_fdr

CRS_M = 'EPSG:32638'
os.makedirs(f'{OUT}/figures', exist_ok=True)
FIG = f'{OUT}/figures'

def make_grid(poly_m, res, buf):
    x0, y0, x1, y1 = poly_m.buffer(buf).bounds
    x0, y0 = math.floor(x0 / 30) * 30, math.floor(y0 / 30) * 30
    x1, y1 = math.ceil(x1 / 30) * 30, math.ceil(y1 / 30) * 30
    return from_origin(x0, y1, res, res), int(round((x1 - x0) / res)), int(round((y1 - y0) / res))

def read_to_grid(path, grid, resampling=Resampling.bilinear, band=1, ovr=None):
    # Reads any (remote) raster straight into our grid; only the needed blocks are downloaded.
    # ovr: overview level to read from (for coarse grids), falls back to full resolution.
    T, W, H = grid
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', AWS_NO_SIGN_REQUEST='YES',
                      GDAL_HTTP_MAX_RETRY='4', GDAL_HTTP_RETRY_DELAY='2'):
        try:
            src = rasterio.open(path, overview_level=ovr) if ovr is not None else rasterio.open(path)
        except Exception:
            src = rasterio.open(path)
        with src, WarpedVRT(src, crs=CRS_M, transform=T, width=W, height=H, resampling=resampling) as vrt:
            a = vrt.read(band, masked=True)
    return a.astype('float32').filled(np.nan)

def to_gpkg(gdf, path):
    # GeoPackage (SQLite) is written locally first, then copied: SQLite files are unreliable on the Drive mount.
    import shutil
    tmp = '/content/_tmp_' + os.path.basename(path)
    if os.path.exists(tmp): os.remove(tmp)
    gdf.to_file(tmp, driver='GPKG'); shutil.copy(tmp, path)

def save_tif(arr, path, grid, nodata=-9999):
    T, W, H = grid
    a = np.where(np.isfinite(arr), arr, nodata).astype('float32')
    with rasterio.open(path, 'w', driver='GTiff', height=H, width=W, count=1, dtype='float32', crs=CRS_M,
                       transform=T, nodata=nodata, compress='deflate', tiled=True) as d:
        d.write(a, 1)

def extent(grid):
    T, W, H = grid
    return [T.c, T.c + W * T.a, T.f + H * T.e, T.f]

def inside_mask(poly_m, grid):
    T, W, H = grid
    return rasterize([(poly_m, 1)], out_shape=(H, W), transform=T, fill=0, dtype='uint8').astype(bool)

def scalebar(ax, length=500):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x, y = x0 + 0.05 * (x1 - x0), y0 + 0.05 * (y1 - y0)
    ax.plot([x, x + length], [y, y], color='k', lw=3); ax.text(x + length / 2, y + 0.02 * (y1 - y0), f'{length} m', ha='center', fontsize=8)
    ax.annotate('N', xy=(0.94, 0.95), xytext=(0.94, 0.84), xycoords='axes fraction', ha='center', fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='-|>', color='k'))

D = {}
print('model ready')

## Step 1 · Boundaries (official Geoportal exports, nothing to upload)
- **Old core** = Al-Dirah + Al-Futah (exports of 4 Nov 2026), about **2.5 km²**. Each street keeps its sub-district in the column `zone`.
- **Al-Waha** = the official Al-Waha outline (export of 30 Oct 2026) inside King Salman District, about **3.1 km²**. Salah al-Din is not included.

Check the map: use the layer button (top right) → **Satellite**.

In [ ]:
UTM = CRS_M
ALDIRAH = [[46.706271, 24.623278], [46.709661, 24.625343], [46.710949, 24.626435], [46.711807, 24.626863], [46.712966, 24.627369], [46.713781, 24.627993], [46.712065, 24.629941], [46.712236, 24.630214], [46.714983, 24.631071], [46.715584, 24.631071], [46.716099, 24.62963], [46.717257, 24.630058], [46.716528, 24.630994], [46.716228, 24.632552], [46.716657, 24.634345], [46.717043, 24.635241], [46.717515, 24.636449], [46.717644, 24.637813], [46.717172, 24.637891], [46.714768, 24.638242], [46.713996, 24.638359], [46.710305, 24.638047], [46.705499, 24.637852], [46.704297, 24.637774], [46.705413, 24.631227], [46.705756, 24.629045], [46.705756, 24.628032], [46.706271, 24.623278]]
ALFUTAH = [[46.70404, 24.63797], [46.711679, 24.638165], [46.713181, 24.638282], [46.71494, 24.638282], [46.71773, 24.637931], [46.717772, 24.640113], [46.717558, 24.641321], [46.715069, 24.646426], [46.707644, 24.644828], [46.705842, 24.644283], [46.703267, 24.644789], [46.703739, 24.640893], [46.703954, 24.63836], [46.70404, 24.63797]]
KSD = [[46.693611, 24.739978], [46.720991, 24.751398], [46.725669, 24.742473], [46.723909, 24.731207], [46.721592, 24.730623], [46.711249, 24.736665], [46.709104, 24.732923], [46.705885, 24.731792], [46.706743, 24.729609], [46.705585, 24.726257], [46.698933, 24.723645], [46.696959, 24.72306], [46.694984, 24.722592], [46.693525, 24.721695], [46.692967, 24.721656], [46.685629, 24.736548], [46.685801, 24.73686], [46.693611, 24.739978]]
ALWAHA = [[46.709035, 24.733092], [46.703438, 24.74422], [46.721157, 24.751547], [46.725619, 24.742544], [46.72386, 24.731397], [46.721372, 24.730656], [46.711161, 24.737009], [46.709035, 24.733092]]

to_m = lambda ring: gpd.GeoSeries([Polygon(ring)], crs=4326).to_crs(UTM).iloc[0].buffer(0)
dd, ff = to_m(ALDIRAH), to_m(ALFUTAH)
core = unary_union([dd.buffer(15), ff.buffer(15)]).buffer(-15)          # close the < 30 m gap along the shared street
core = max(core.geoms, key=lambda p: p.area) if core.geom_type == 'MultiPolygon' else core
fz = ff.intersection(core); dz = core.difference(fz)
waha = to_m(ALWAHA).intersection(to_m(KSD))
ZONES = {'Old core': {'Al-Dirah': dz, 'Al-Futah': fz}, 'Al-Waha': {'Al-Waha': waha}}

for name, poly_m in [('Old core', core), ('Al-Waha', waha)]:
    poly_wgs = gpd.GeoSeries([poly_m], crs=UTM).to_crs(4326).iloc[0]
    c = poly_wgs.centroid
    D[name] = dict(poly_m=poly_m, poly_wgs=poly_wgs, lat=c.y, lon=c.x, grid=make_grid(poly_m, RES, BUFFER_M), dir=f'{OUT}/{name.replace(" ", "_")}')
    os.makedirs(D[name]['dir'], exist_ok=True)
    gpd.GeoDataFrame({'name': [name]}, geometry=[poly_m], crs=UTM).to_file(f"{D[name]['dir']}/boundary.geojson", driver='GeoJSON')
    print(f"{name}: {poly_m.area/1e6:.2f} km² | grid {D[name]['grid'][1]} x {D[name]['grid'][2]} px | centre {c.y:.4f}N {c.x:.4f}E")

import folium
m = folium.Map(tiles='OpenStreetMap')
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attr='Esri', name='Satellite').add_to(m)
for n, z in [(k, v) for zz in ZONES.values() for k, v in zz.items()]:
    folium.GeoJson(gpd.GeoSeries([z], crs=UTM).to_crs(4326).__geo_interface__, name=n,
                   style_function=lambda f, c={'Al-Dirah': '#9e2a1f', 'Al-Futah': '#e08a3c', 'Al-Waha': '#3c78b4'}[n]: {'color': c, 'weight': 3, 'fillOpacity': 0.1}).add_to(m)
x0, y0, x1, y1 = gpd.GeoSeries([core, waha], crs=UTM).to_crs(4326).total_bounds
m.fit_bounds([[y0, x0], [y1, x1]]); folium.LayerControl().add_to(m); display(m)

## Step 2 · Buildings and heights (≈3–8 min; reused if already in Drive)
Overture Maps footprints; heights from mapped heights, floors × 3.3 m, or GlobalBuildingAtlas, in that order (column `height_src`).

In [ ]:
import duckdb

def duck_query(src_sql, bbox, cols):
    W, S, E, N = bbox
    con = duckdb.connect()
    con.execute("INSTALL spatial; LOAD spatial; INSTALL httpfs; LOAD httpfs; SET s3_region='us-west-2';")
    where = f'bbox.xmin < {E} AND bbox.xmax > {W} AND bbox.ymin < {N} AND bbox.ymax > {S}'
    sel = ', '.join(cols)
    try:
        df = con.execute(f'SELECT {sel}, ST_AsWKB(geometry) AS wkb FROM {src_sql} WHERE {where}').df()
    except Exception:
        df = con.execute(f'SELECT {sel}, geometry AS wkb FROM {src_sql} WHERE {where}').df()
    return gpd.GeoDataFrame(df.drop(columns='wkb'), geometry=shapely.from_wkb([bytes(b) for b in df['wkb']]), crs=4326)

def overture_buildings(bbox, out):
    W, S, E, N = bbox
    try:
        if not os.path.exists(out):
            subprocess.run(['overturemaps', 'download', f'--bbox={W},{S},{E},{N}', '-f', 'geoparquet', '--type=building', '-o', out],
                           check=True, capture_output=True, text=True)
        g = gpd.read_parquet(out).set_crs(4326, allow_override=True); src = 'Overture Maps (overturemaps tool)'
    except Exception:
        print('   overturemaps tool failed -> direct read of release', OVERTURE_RELEASE)
        g = duck_query(f"read_parquet('s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=buildings/type=building/*', hive_partitioning=1)",
                       bbox, ['id', 'height', 'num_floors'])
        src = f'Overture Maps release {OVERTURE_RELEASE}'
    for c in ('height', 'num_floors'):
        if c not in g: g[c] = np.nan
    return g[['id', 'height', 'num_floors', 'geometry']], src

def gba_buildings(bbox):
    W, S, E, N = bbox
    names = set()
    for x in (W, E):
        for y in (S, N):
            w, s = 5 * math.floor(x / 5), 5 * math.floor(y / 5)
            names.add(f"{'e' if w >= 0 else 'w'}{abs(w):03d}_n{s + 5:02d}_{'e' if w + 5 >= 0 else 'w'}{abs(w + 5):03d}_n{s:02d}.parquet")
    parts = []
    for t in sorted(names):
        try:
            parts.append(duck_query(f"read_parquet('https://data.source.coop/tge-labs/globalbuildingatlas-lod1/{t}')", bbox, ['height']))
            print(f'   GlobalBuildingAtlas {t}: {len(parts[-1]):,} buildings')
        except Exception as e:
            print(f'   GlobalBuildingAtlas {t} not read ({str(e)[:80]})')
    return pd.concat(parts, ignore_index=True) if parts else None

def assign_heights(ov, gba):
    ov = ov.to_crs(CRS_M).reset_index(drop=True)
    ov = ov[ov.geometry.area >= 10].reset_index(drop=True); ov['bid'] = ov.index
    ov['h'] = np.nan; ov['height_src'] = None
    s = ov['height'].notna() & (ov['height'] > 0); ov.loc[s, 'h'] = ov.loc[s, 'height']; ov.loc[s, 'height_src'] = 'overture_h'
    s = ov['h'].isna() & ov['num_floors'].notna() & (ov['num_floors'] > 0)
    ov.loc[s, 'h'] = ov.loc[s, 'num_floors'] * FLOOR_H; ov.loc[s, 'height_src'] = 'overture_floors'
    added = 0
    if gba is not None and len(gba):
        g = gba.to_crs(CRS_M); g = g[g['height'].notna() & (g['height'] > 0)][['height', 'geometry']].rename(columns={'height': 'gba_h'})
        g = g[g.geometry.is_valid]
        inter = gpd.overlay(ov[['bid', 'geometry']], g, how='intersection', keep_geom_type=True); inter['a'] = inter.geometry.area
        best = inter.sort_values('a').groupby('bid').tail(1).set_index('bid')['gba_h']
        s = ov['h'].isna() & ov['bid'].isin(best.index); ov.loc[s, 'h'] = ov.loc[s, 'bid'].map(best); ov.loc[s, 'height_src'] = 'gba'
        j = gpd.sjoin(g, ov[['geometry']], how='left', predicate='intersects')
        only = g.loc[j[j['index_right'].isna()].index.unique()]; only = only[only.geometry.area >= 20]
        if len(only):
            ov = pd.concat([ov, gpd.GeoDataFrame({'h': only['gba_h'].values, 'height_src': 'gba_only'}, geometry=only.geometry.values, crs=CRS_M)], ignore_index=True)
            added = len(only)
    s = ov['h'].isna(); ov.loc[s, 'h'] = DEFAULT_H; ov.loc[s, 'height_src'] = 'default'
    ov['h'] = ov['h'].clip(2.5, 300)
    return gpd.GeoDataFrame(ov[['h', 'height_src', 'geometry']], crs=CRS_M), added

for name, d in D.items():
    gp = f"{d['dir']}/buildings_with_heights.gpkg"
    if os.path.exists(gp):
        d['bldg'] = gpd.read_file(gp).to_crs(CRS_M); d['bldg_src'] = open(gp + '.src').read() if os.path.exists(gp + '.src') else 'Overture Maps'
        print(f"{name}: reused {len(d['bldg']):,} buildings from Drive"); continue
    bbox = tuple(gpd.GeoSeries([d['poly_m'].buffer(BUFFER_M)], crs=CRS_M).to_crs(4326).total_bounds)
    ov, src = overture_buildings(bbox, f"/content/{name.replace(' ', '_')}_overture.parquet")
    print(f'{name}: Overture {len(ov):,} footprints')
    b, added = assign_heights(ov, gba_buildings(bbox))
    d['bldg'], d['bldg_src'] = b, src
    to_gpkg(b, gp); open(gp + '.src', 'w').write(src)
    print(f"   {len(b):,} buildings ({added} added from GBA) | height sources %: {b['height_src'].value_counts(normalize=True).mul(100).round(1).to_dict()}")
    print(f"   median height {b['h'].median():.1f} m | 95th percentile {b['h'].quantile(.95):.1f} m")

## Step 3 · Trees, ground and land cover (≈2–5 min)
Meta/WRI 1 m canopy height (trees ≥ 3 m), Copernicus GLO-30 DEM and ESA WorldCover 10 m (albedo and paved ground), all without an account.

In [ ]:
import pystac_client, planetary_computer, random

def pc_client(tries=8):
    # Planetary Computer sometimes rate-limits Colab; wait and retry (up to ~15 min) instead of failing.
    for k in range(tries):
        try:
            return pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace)
        except Exception as e:
            wait = min(240, 15 * 2 ** k) + random.uniform(0, 5)
            print(f'   Planetary Computer busy ({str(e)[:60]}) - waiting {wait:.0f} s'); time.sleep(wait)
    raise RuntimeError('Planetary Computer not reachable now. Wait 30-60 minutes, then re-run this cell.')

def pc_items(collections, bbox, datetime=None, query=None, tries=6):
    for k in range(tries):
        try:
            return list(pc_client().search(collections=collections, bbox=list(bbox), datetime=datetime, query=query).items())
        except Exception as e:
            wait = min(240, 15 * 2 ** k); print(f'   search busy ({str(e)[:60]}) - waiting {wait} s'); time.sleep(wait)
    raise RuntimeError('Planetary Computer search failed repeatedly. Wait 30-60 minutes, then re-run this cell.')

def pc_to_grid(collection, asset, bbox, grid, resampling, newest_first=False, ovr=None):
    items = pc_items([collection], bbox)
    if newest_first:
        items.sort(key=lambda i: str(i.properties.get('start_datetime') or i.datetime), reverse=True)
    out = np.full((grid[2], grid[1]), np.nan, 'float32')
    for it in items:
        out = np.where(np.isnan(out), read_to_grid(it.assets[asset].href, grid, resampling, ovr=ovr), out)
    return out

# Public copies on Amazon (no account, no rate limit): Copernicus GLO-30 DEM and ESA WorldCover 2021
def aws_dem(bbox, grid, resampling=Resampling.bilinear, ovr=None):
    W_, S_, E_, N_ = bbox; out = np.full((grid[2], grid[1]), np.nan, 'float32')
    for la in range(math.floor(S_), math.floor(N_) + 1):
        for lo in range(math.floor(W_), math.floor(E_) + 1):
            nm = f'Copernicus_DSM_COG_10_N{la:02d}_00_E{lo:03d}_00_DEM'
            try: out = np.where(np.isnan(out), read_to_grid(f'https://copernicus-dem-30m.s3.amazonaws.com/{nm}/{nm}.tif', grid, resampling, ovr=ovr), out)
            except Exception as e: print('   DEM tile', nm, 'not read:', str(e)[:60])
    return out

def aws_worldcover(bbox, grid, resampling=Resampling.nearest, ovr=None):
    W_, S_, E_, N_ = bbox; out = np.full((grid[2], grid[1]), np.nan, 'float32')
    for la in range(3 * math.floor(S_ / 3), 3 * math.floor(N_ / 3) + 1, 3):
        for lo in range(3 * math.floor(W_ / 3), 3 * math.floor(E_ / 3) + 1, 3):
            u = f'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N{la:02d}E{lo:03d}_Map.tif'
            try: out = np.where(np.isnan(out), read_to_grid(u, grid, resampling, ovr=ovr), out)
            except Exception as e: print('   WorldCover tile not read:', str(e)[:60])
    return out

def get_dem(bbox, grid, resampling=Resampling.bilinear, ovr=None):
    a = aws_dem(bbox, grid, resampling, ovr)
    if np.isnan(a).all(): a = pc_to_grid('cop-dem-glo-30', 'data', bbox, grid, resampling, ovr=ovr)
    return a

def get_worldcover(bbox, grid, resampling=Resampling.nearest, ovr=None):
    a = aws_worldcover(bbox, grid, resampling, ovr)
    if np.isnan(a).all(): a = pc_to_grid('esa-worldcover', 'map', bbox, grid, resampling, newest_first=True, ovr=ovr)
    return a

CHM_BASE = 'https://dataforgood-fb-data.s3.amazonaws.com/forests/v1/alsgedi_global_v6_float'
try:
    chm_tiles = gpd.read_file(f'{CHM_BASE}/tiles.geojson')
    tcol = 'tile' if 'tile' in chm_tiles else [c for c in chm_tiles.columns if c != 'geometry'][0]
except Exception as e:
    print('canopy tile index not read:', str(e)[:80]); chm_tiles = gpd.GeoDataFrame(geometry=[], crs=4326); tcol = 'tile'

WC_ALBEDO = {10: 0.15, 20: 0.20, 30: 0.20, 40: 0.20, 50: 0.18, 60: 0.30, 80: 0.06, 90: 0.15, 95: 0.15, 100: 0.25}
for name, d in D.items():
    g = d['grid']; T, W, H = g
    f = {k: f"{d['dir']}/{k}_{RES:g}m.tif" for k in ('CDSM_relative', 'DEM', 'landcover')}
    ok = all(os.path.exists(x) for x in f.values())
    if ok:   # reuse only if the saved ground and land cover are real (not the flat/uniform fallbacks of a failed download)
        dem_ = read_to_grid(f['DEM'], g); wc_ = read_to_grid(f['landcover'], g, Resampling.nearest)
        ok = (np.nanmax(dem_) - np.nanmin(dem_) > 0.5) and (len(np.unique(wc_[np.isfinite(wc_)])) > 1)
    if ok:
        c = read_to_grid(f['CDSM_relative'], g, Resampling.nearest)
        d.update(can_raw=np.where(c > 0, c, np.nan).astype('float32'), dem=dem_, wc=wc_)
        print(f'{name}: reused trees, ground and land cover from Drive')
    else:
        bbox = tuple(gpd.GeoSeries([d['poly_m'].buffer(BUFFER_M + 50)], crs=CRS_M).to_crs(4326).total_bounds)
        can = np.full((H, W), np.nan, 'float32')
        try:
            for t in chm_tiles[chm_tiles.intersects(box(*bbox))][tcol]:
                can = np.fmax(can, read_to_grid(f'/vsicurl/{CHM_BASE}/chm/{t}.tif', g, Resampling.nearest))
        except Exception as e:
            print('   canopy map not read -> no trees; re-run this cell later', str(e)[:80])
        dem = get_dem(bbox, g, Resampling.bilinear); dem = np.where(np.isnan(dem), np.nanmean(dem), dem)
        wc = get_worldcover(bbox, g, Resampling.nearest)
        if np.isnan(dem).all() or np.isnan(wc).all():
            raise RuntimeError('Ground elevation or land cover could not be downloaded. Re-run this cell in a few minutes.')
        d.update(can_raw=can, dem=dem.astype('float32'), wc=wc)
        save_tif(np.nan_to_num(can), f['CDSM_relative'], g); save_tif(dem, f['DEM'], g); save_tif(wc, f['landcover'], g)
    d['albedo'] = np.vectorize(lambda v: WC_ALBEDO.get(int(v), 0.20) if np.isfinite(v) else 0.20)(d['wc']).astype('float32')
    print(f"{name}: canopy ≥ 3 m on {np.nanmean(d['can_raw'] >= 3)*100:.2f}% | ground {np.nanmin(d['dem']):.0f}–{np.nanmax(d['dem']):.0f} m")

## Step 4 · Weather for the design day
ERA5 via Open-Meteo, at the midpoint of the two districts (they are 12 km apart, inside the same weather cells), so **both districts get the same weather**: differences in pedestrian heat then come from urban form only.

In [ ]:
def openmeteo(lat, lon, start, end):
    base = 'https://archive-api.open-meteo.com/v1/archive'
    common = 'temperature_2m,relative_humidity_2m,wind_speed_10m,'
    for rad, tag in [('shortwave_radiation_instant,direct_normal_irradiance_instant,diffuse_radiation_instant', 'instantaneous'),
                     ('shortwave_radiation,direct_normal_irradiance,diffuse_radiation', 'preceding-hour mean')]:
        for attempt in range(4):
            r = requests.get(base, params=dict(latitude=lat, longitude=lon, start_date=start, end_date=end, hourly=common + rad,
                                               timezone='Asia/Riyadh', wind_speed_unit='ms'), timeout=90)
            if r.status_code == 429: time.sleep(30); continue
            break
        if r.ok and 'hourly' in r.json(): break
    r.raise_for_status()
    df = pd.DataFrame(r.json()['hourly']); df['time'] = pd.to_datetime(df['time']); df = df.set_index('time')
    df = df.rename(columns={c: ('ta' if c.startswith('temperature') else 'rh' if c.startswith('relative') else 'ws' if c.startswith('wind')
                                else 'ghi' if c.startswith('shortwave') else 'dni' if c.startswith('direct_normal') else 'dhi') for c in df.columns}).astype(float)
    df['dni'] = df['dni'].clip(0, 1100); df['ghi'] = df['ghi'].clip(lower=0); df['dhi'] = df['dhi'].clip(lower=0)
    df.attrs['radiation'] = tag
    return df

lat0 = np.mean([d['lat'] for d in D.values()]); lon0 = np.mean([d['lon'] for d in D.values()])
season = openmeteo(lat0, lon0, f'{WEATHER_YEAR}-07-01', f'{WEATHER_YEAR}-08-31')
daily = season.resample('D').agg(tmax=('ta', 'max'), ghi_sum=('ghi', 'sum'))
DAY = (daily[daily.ghi_sum >= 0.95 * daily.ghi_sum.max()].tmax.idxmax().date() if DESIGN_DAY == 'auto' else pd.Timestamp(DESIGN_DAY).date())
WX = season[season.index.date == DAY]
WX.to_csv(f'{OUT}/design_day_weather.csv')
print('Design day:', DAY, '| radiation:', season.attrs['radiation'])
display(WX.loc[WX.index.hour.isin(HOURS)].round(1))

## Step 5 · 3D surface model (1 m)

In [ ]:
for name, d in D.items():
    g = d['grid']; T, W, H = g; b = d['bldg']
    bh = rasterize(((gm, h) for gm, h in zip(b.geometry, b['h'])), out_shape=(H, W), transform=T, fill=0, dtype='float32')
    fp = rasterize(((gm, gm.area) for gm in b.geometry), out_shape=(H, W), transform=T, fill=0, dtype='float32')   # footprint area
    can = np.where((d['can_raw'] >= 3) & (bh == 0), d['can_raw'], np.nan).astype('float32')
    d.update(bh=bh, fp=np.where(bh > 0, fp, np.nan), obst=bh, ground=bh == 0, can=np.nan_to_num(can),
             dsm=(d['dem'] + bh).astype('float32'), veg_top=(d['dem'] + can).astype('float32'), veg_base=(d['dem'] + 0.25 * can).astype('float32'))
    d['inside'] = inside_mask(d['poly_m'], g)
    for k, a in [('DSM', d['dsm']), ('building_height', bh)]:
        save_tif(a, f"{d['dir']}/{k}_{RES:g}m.tif", g)
    ins = d['inside']
    print(f"{name}: building cover {np.mean(bh[ins] > 0)*100:.1f}% | mean height {bh[ins & (bh > 0)].mean():.1f} m | trees {np.mean(d['can'][ins] > 0)*100:.2f}%")

## Step 6 · Streets and pedestrian space (measured widths)
The old core has lanes of 3–8 m between buildings, the villa district streets of 15–30 m. One rule set is applied to both:
1. The **width of every street is measured** from the buildings (median facade-to-facade distance along the street).
2. **Shared lanes** (narrower than 8 m): the whole lane is pedestrian space.
3. **Pedestrian streets and plazas**: a band of up to 4 m each side of the line; footpaths 2.5 m wide.
4. **Streets with traffic**: 2.5 m sidewalks outside the carriageway; the carriageway is the class width, narrowed where the facades are closer.

H/W uses the measured width and the mean building height in the 20 m beyond it.

In [ ]:
import osmnx as ox
ox.settings.log_console = False; ox.settings.requests_timeout = 300
HALF = {'trunk': 16, 'primary': 14, 'secondary': 11, 'tertiary': 8, 'residential': 6, 'unclassified': 6, 'living_street': 4,
        'service': 3, 'pedestrian': 0, 'footway': 0, 'path': 0, 'steps': 0}
SKIP_SERVICE = {'parking_aisle', 'driveway', 'drive-through', 'emergency_access'}
OVERPASS = ['https://overpass-api.de/api', 'https://overpass.kumi.systems/api', 'https://overpass.private.coffee/api']

def first(v):
    return v[0] if isinstance(v, list) else v

def osm_streets(poly_wgs):
    last = None
    for url in OVERPASS:
        ox.settings.overpass_url = url
        try: ox.settings.overpass_endpoint = url
        except Exception: pass
        try:
            G = ox.graph_from_polygon(poly_wgs.buffer(0.001), network_type='all', retain_all=True, truncate_by_edge=True)
            try: Gu = ox.convert.to_undirected(G)
            except AttributeError: Gu = ox.get_undirected(G)
            e = ox.graph_to_gdfs(Gu, nodes=False).reset_index(drop=True)
            e['cls'] = e['highway'].apply(first).astype(str).str.replace('_link', '')
            e['svc'] = e['service'].apply(first).astype(str) if 'service' in e else ''
            return e[['cls', 'svc', 'geometry']].to_crs(CRS_M), f'OpenStreetMap ({url})'
        except Exception as ex:
            last = ex; print(f'   {url} failed: {str(ex)[:80]}')
    raise RuntimeError(last)

def overture_streets(poly_wgs, out):
    W, S, E, N = poly_wgs.buffer(0.001).bounds
    if not os.path.exists(out):
        subprocess.run(['overturemaps', 'download', f'--bbox={W},{S},{E},{N}', '-f', 'geoparquet', '--type=segment', '-o', out], check=True, capture_output=True, text=True)
    s = gpd.read_parquet(out).set_crs(4326, allow_override=True)
    if 'subtype' in s: s = s[s['subtype'] == 'road']
    s['cls'] = s['class'].astype(str).replace({'sidewalk': 'footway', 'crosswalk': 'footway'})
    s['svc'] = s['subclass'].astype(str) if 'subclass' in s else ''
    s = s[s.geometry.geom_type.isin(['LineString', 'MultiLineString'])].explode(index_parts=False)
    return s[['cls', 'svc', 'geometry']].to_crs(CRS_M), 'Overture Maps transportation'

def bearing(gm):
    (x0, y0), (x1, y1) = gm.coords[0], gm.coords[-1]
    return math.degrees(math.atan2(x1 - x0, y1 - y0)) % 180

for name, d in D.items():
    g = d['grid']; T, W, H = g
    cache = f"{d['dir']}/streets_{STREET_SOURCE}.gpkg"
    if os.path.exists(cache):
        e = gpd.read_file(cache).to_crs(CRS_M); src = open(cache + '.src').read()
    else:
        if STREET_SOURCE == 'overture':
            e, src = overture_streets(d['poly_wgs'], f"/content/{name.replace(' ', '_')}_segments.parquet")
        else:
            e, src = osm_streets(d['poly_wgs'])
        to_gpkg(e, cache); open(cache + '.src', 'w').write(src)
    e = e[e['cls'].isin(HALF) & ~e['svc'].isin(SKIP_SERVICE)].reset_index(drop=True)
    e = e[e.geometry.length >= 12]
    e = e[e.geometry.interpolate(0.5, normalized=True).within(d['poly_m'])].reset_index(drop=True)
    e['seg'] = np.arange(1, len(e) + 1)
    e['half_class'] = e['cls'].map(HALF)
    free = distance_transform_edt(d['bh'] == 0) * RES                       # distance to the nearest building (m)
    e['half_w_meas'] = measured_half_width(e, free, T, RES)
    geo = [realm_geometry(gm, c, hc, hw, SW_W, SHARED_LANE_W, plaza_half=(4.0 if c == 'pedestrian' else SW_W / 2))
           for gm, c, hc, hw in zip(e.geometry, e['cls'], e['half_class'], e['half_w_meas'])]
    e['street_w'] = [x[2] for x in geo]; e['realm'] = [x[3] for x in geo]
    lab = rasterize(((x[0], int(i)) for x, i in zip(geo, e['seg']) if not x[0].is_empty), out_shape=(H, W), transform=T, fill=0, dtype='int32')
    carr = rasterize(((x[1], 1) for x in geo if x[1] is not None), out_shape=(H, W), transform=T, fill=0, dtype='uint8').astype(bool)
    lab[~d['ground'] | carr] = 0
    d['labels'], d['carriage'], d['sidewalk'] = lab, carr & d['ground'], lab > 0
    hwm = np.where(np.isfinite(e['half_w_meas']), e['half_w_meas'], e['half_class'] + SW_W)
    front = [gm.buffer(h + 20, cap_style=2).difference(gm.buffer(h, cap_style=2)) for gm, h in zip(e.geometry, hwm)]
    flab = rasterize(((gm, int(i)) for gm, i in zip(front, e['seg'])), out_shape=(H, W), transform=T, fill=0, dtype='int32')
    n = len(e)
    e['bldg_h_front'] = np.nan_to_num(zonal_mean(flab, np.where(d['bh'] > 0, d['bh'], np.nan), n)[1:])
    e['hw_ratio'] = e['bldg_h_front'] / e['street_w'].clip(lower=2)
    e['canopy_pct'] = 100 * zonal_mean(lab, (d['can'] > 0).astype('float32'), n)[1:]
    e['ped_m2'] = np.bincount(lab.ravel(), minlength=n + 1)[1:] * RES * RES
    e['length_m'] = e.geometry.length
    e['bearing'] = e.geometry.apply(bearing)
    e['orient'] = pd.cut(e['bearing'], [-1, 22.5, 67.5, 112.5, 157.5, 181], labels=['N-S', 'NE-SW', 'E-W', 'NW-SE', 'N-S2']).astype(str).replace('N-S2', 'N-S')
    mid = e.geometry.interpolate(0.5, normalized=True)
    e['zone'] = name
    for zn, zp in ZONES[name].items():
        e.loc[mid.within(zp).values, 'zone'] = zn
    d['seg'], d['n_seg'], d['street_src'] = e, n, src
    print(f"{name}: {n} segments ({src}) | {e['length_m'].sum()/1000:.1f} km | pedestrian space {e['ped_m2'].sum()/1e4:.1f} ha")
    display(e.groupby('realm').agg(segments=('seg', 'size'), km=('length_m', lambda s: round(s.sum() / 1000, 1)),
                                   median_width_m=('street_w', 'median'), median_HW=('hw_ratio', 'median')).round(2))

## Step 7 · Sky view factor (≈2–4 min per district)

In [ ]:
for name, d in D.items():
    t = time.time()
    d['svf'] = sky_view_factor(d['dsm'], RES, d['veg_top'], n_az=24)
    save_tif(np.where(d['ground'], d['svf'][1], np.nan), f"{d['dir']}/SVF_{RES:g}m.tif", d['grid'])
    d['seg']['svf'] = zonal_mean(d['labels'], d['svf'][1], d['n_seg'])[1:]
    print(f"{name}: {time.time()-t:.0f} s | median SVF of pedestrian space {np.nanmedian(d['seg']['svf']):.2f}")

## Step 8 · Hourly shade, Tmrt and UTCI on the design day (≈3–8 min per district)
UTCI heat-stress classes: 32–38 °C strong, 38–46 °C very strong, above 46 °C extreme.

In [ ]:
from pythermalcomfort.models import utci as _utci
def utci_fn(ta, tmrt, ws, rh):
    out = np.full(tmrt.shape, np.nan, 'float32'); ok = np.isfinite(tmrt)
    tr = tmrt[ok].astype('float64'); res = np.empty_like(tr); v = float(np.clip(ws, 0.5, 17.0))
    for i in range(0, tr.size, 500_000):
        c = tr[i:i + 500_000]; n = c.size
        kw = dict(tdb=np.full(n, float(ta)), tr=c, v=np.full(n, v), rh=np.full(n, float(rh)))
        try: r = _utci(**kw, limit_inputs=False)
        except TypeError: r = _utci(**kw)
        r = getattr(r, 'utci', r)
        if isinstance(r, dict): r = r['utci']
        res[i:i + n] = np.asarray(r, dtype='float64')
    out[ok] = res
    return out

SEG_H, DIST_H = [], []
for name, d in D.items():
    print(name)
    seg_h, dist_h, daily_r, key = run_day(DAY, WX, d['dsm'], d['veg_top'], d['veg_base'], d['ground'], d['svf'], d['labels'], d['n_seg'],
                                          d['lat'], d['lon'], UTC_OFFSET, HOURS, RES, utci_fn=utci_fn, albedo_g=d['albedo'])
    d['daily'], d['key'] = daily_r, key
    d['seg'] = d['seg'].merge(segment_summary(seg_h, daily_r, d['labels'], d['n_seg']), left_on='seg', right_index=True, how='left')
    seg_h['district'] = name; dist_h['district'] = name; SEG_H.append(seg_h); DIST_H.append(dist_h)
    for k, a in daily_r.items(): save_tif(np.where(d['ground'], a, np.nan), f"{d['dir']}/{k}_{DAY}.tif", d['grid'])
    for h, rr in key.items():
        for k, a in rr.items(): save_tif(a, f"{d['dir']}/{k}_{h:02d}00_{DAY}.tif", d['grid'])
SEG_H = pd.concat(SEG_H, ignore_index=True); DIST_H = pd.concat(DIST_H, ignore_index=True)
SEG_H.to_csv(f'{OUT}/segments_hourly.csv', index=False); DIST_H.to_csv(f'{OUT}/districts_hourly.csv', index=False)

## Step 9 · Local Climate Zones (100 m grid)
Each 100 m cell gets the Stewart & Oke (2012) zone whose value ranges best match its sky view factor, aspect ratio, building surface fraction, paved and unpaved fractions and building height. LCZ 8 (large low-rise) is only allowed where buildings are large (mean footprint ≥ 1000 m²). Each cell also gets its mean pedestrian heat (UTCI at 15:00, hours ≥ 46 °C).

In [ ]:
LCZ = []
for name, d in D.items():
    imperv = (d['wc'] == 50)                                 # WorldCover built-up (paved) ground
    cells = morphology_cells(d['bh'], d['can'], d['svf'][1], imperv, d['ground'], d['grid'], LCZ_CELL, fp_px=d['fp'])
    f = int(round(LCZ_CELL / RES)); H2, W2 = (d['grid'][2] // f) * f, (d['grid'][1] // f) * f
    agg = lambda a: np.nanmean(a[:H2, :W2].reshape(H2 // f, f, W2 // f, f), axis=(1, 3)).ravel()
    cells['inside'] = agg(d['inside'].astype('float32')) >= 0.5
    cells['utci_15'] = agg(np.where(d['ground'], d['key'][15]['utci'], np.nan))
    cells['utci_ge46_hours'] = agg(np.where(d['ground'], d['daily']['utci_ge46_hours'], np.nan))
    cells['tmrt_mean'] = agg(np.where(d['ground'], d['daily']['tmrt_mean'], np.nan))
    cells['shade_15'] = 100 * agg(np.where(d['ground'], d['key'][15]['shade'], np.nan))
    cells['district'] = name
    cells = cells[cells.inside].copy(); d['cells'] = cells; LCZ.append(cells)
    print(name); display(cells.lcz.value_counts(normalize=True).mul(100).round(1).rename('% of district').to_frame())
LCZ = pd.concat(LCZ, ignore_index=True)
to_gpkg(gpd.GeoDataFrame(LCZ, geometry=[box(x - LCZ_CELL / 2, y - LCZ_CELL / 2, x + LCZ_CELL / 2, y + LCZ_CELL / 2) for x, y in zip(LCZ.x, LCZ.y)],
                         crs=CRS_M), f'{OUT}/lcz_cells.gpkg')
LCZ.to_csv(f'{OUT}/lcz_cells.csv', index=False)
display(LCZ.groupby(['district', 'lcz'])[['bsf', 'h', 'svf', 'ar', 'utci_15', 'utci_ge46_hours']].median().round(2))

## Step 10 · Surface heat islands at 30 m: Landsat LST and hot spots (≈5–10 min)
- Summer (June–September) median Landsat 8/9 surface temperature, overpass about 10:30, on each district **plus a 1.5 km ring**.
- **Context contrast** = district mean LST − ring mean LST.
- **Hot and cold spots**: Getis-Ord Gi* (120 m neighbourhood) with false-discovery-rate control; classes at 90, 95 and 99% confidence.

In [ ]:
def landsat_lst(poly_m, buf):
    g30 = make_grid(poly_m, 30, buf)
    bbox = list(gpd.GeoSeries([poly_m.buffer(buf)], crs=CRS_M).to_crs(4326).total_bounds)
    items = pc_items(['landsat-c2-l2'], bbox, datetime=f'{LST_YEARS[0]}-01-01/{LST_YEARS[1]}-12-31',
                     query={'eo:cloud_cover': {'lt': LST_MAX_CLOUD}, 'platform': {'in': ['landsat-8', 'landsat-9']}})
    items = [i for i in items if LST_MONTHS[0] <= i.datetime.month <= LST_MONTHS[1]]
    stack = []
    for it in items:
        try:
            st = read_to_grid(it.assets['lwir11'].href, g30); qa = read_to_grid(it.assets['qa_pixel'].href, g30, Resampling.nearest)
        except Exception as e:
            print('   skipped', it.id, str(e)[:60]); continue
        q = np.nan_to_num(qa, nan=1).astype('int64')
        lst = st * 0.00341802 + 149.0 - 273.15
        lst[~np.isfinite(st) | (st <= 0) | ((q & (1 | 2 | 8 | 16)) > 0)] = np.nan
        if np.mean(np.isfinite(lst)) > 0.5: stack.append(lst)
    return np.nanmedian(np.stack(stack), 0).astype('float32'), g30, len(stack)

def gi_classes(z, sig99, sig95, sig90):
    c = np.zeros(z.shape, 'int8')
    for s, v in ((sig90, 1), (sig95, 2), (sig99, 3)):
        c = np.where(s & (z > 0), v, np.where(s & (z < 0), -v, c))
    return c

LST_ROWS = []
for name, d in D.items():
    try:
        lst, g30, n = landsat_lst(d['poly_m'], CONTEXT_M)
    except Exception as e:
        print(f'{name}: Landsat not read ({str(e)[:100]}); re-run this cell'); d['lst30'] = None; continue
    save_tif(lst, f"{d['dir']}/LST_summer_median_30m_context.tif", g30)
    ins = inside_mask(d['poly_m'], g30)
    ring = inside_mask(d['poly_m'].buffer(CONTEXT_M), g30) & ~inside_mask(d['poly_m'].buffer(60), g30)
    z, p = getis_ord_star(lst, GI_RADIUS_M / 30)
    cls = gi_classes(z, bh_fdr(p, .01), bh_fdr(p, .05), bh_fdr(p, .10))
    save_tif(z, f"{d['dir']}/LST_GiStar_z_30m.tif", g30); save_tif(cls.astype('float32'), f"{d['dir']}/LST_hotspot_class_30m.tif", g30)
    d.update(lst30=lst, g30=g30, gi=cls, lst_n=n)
    f = int(round(30 / RES)); T, W, H = d['grid']
    x0 = int(round((T.c - g30[0].c) / 30)); y0 = int(round((g30[0].f - T.f) / 30))
    sub = lst[y0:y0 + int(np.ceil(H / f)), x0:x0 + int(np.ceil(W / f))]
    d['seg']['lst'] = zonal_mean(d['labels'], np.repeat(np.repeat(sub, f, 0), f, 1)[:H, :W], d['n_seg'])[1:]
    row = {'district': name, 'scenes': n, 'LST_district_mean': np.nanmean(lst[ins]), 'LST_ring_mean': np.nanmean(lst[ring]),
           'contrast_district_minus_ring': np.nanmean(lst[ins]) - np.nanmean(lst[ring]),
           'hot_spot_99_pct_area': 100 * np.mean(cls[ins] == 3), 'hot_spot_any_pct_area': 100 * np.mean(cls[ins] > 0),
           'cold_spot_99_pct_area': 100 * np.mean(cls[ins] == -3), 'cold_spot_any_pct_area': 100 * np.mean(cls[ins] < 0)}
    LST_ROWS.append(row); print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
LST_TABLE = pd.DataFrame(LST_ROWS); LST_TABLE.to_csv(f'{OUT}/table_landsat_lst_hotspots.csv', index=False)

## Step 11 · City heat island by day and night: MODIS Aqua 1 km (≈10–20 min)
- MODIS Aqua 8-day LST (MYD11A2, v6.1) for June–September of every year in `MODIS_YEARS`: **day ≈ 13:30**, **night ≈ 01:30**.
- **Rural reference**: desert pixels 40–70 km from the city centre (bare, sparse or shrub land cover, no built-up land within 3 km, ground within ±150 m of the city's elevation).
- **Surface heat-island intensity (SUHI)** = district LST − median rural LST, per year, by day and by night.

In [ ]:
def modis_grid(lat, lon, half_km=80, res=1000):
    c = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs=4326).to_crs(CRS_M).iloc[0]
    T = from_origin(math.floor((c.x - half_km * 1000) / res) * res, math.ceil((c.y + half_km * 1000) / res) * res, res, res)
    return (T, 2 * half_km, 2 * half_km), c

def asset_key(item, *must):
    for k in item.assets:
        if all(m.lower() in k.lower() for m in must): return k
    return None

def modis_summers():
    grid, centre = modis_grid(*CITY_CENTRE)
    bbox = list(gpd.GeoSeries([box(*gpd.GeoSeries([centre], crs=CRS_M).buffer(80000).total_bounds)], crs=CRS_M).to_crs(4326).total_bounds)
    years = range(MODIS_YEARS[0], MODIS_YEARS[1] + 1)
    out = {}
    for y in years:
        items = pc_items(['modis-11A2-061'], bbox, datetime=f'{y}-{MODIS_MONTHS[0]:02d}-01/{y}-{MODIS_MONTHS[1]:02d}-30')   # fresh signed links every year
        items = [i for i in items if i.id.upper().startswith('MYD') or str(i.properties.get('platform', '')).lower() == 'aqua']
        if not items:
            print(f'{y}: no Aqua items found'); continue
        acc = {'day': [], 'night': []}
        for it in items:
            for part, key in (('day', asset_key(it, 'lst_day')), ('night', asset_key(it, 'lst_night'))):
                if key is None: continue
                try:
                    a = read_to_grid(it.assets[key].href, grid, Resampling.nearest)
                except Exception:
                    continue
                qk = asset_key(it, 'qc_' + part)
                if qk:
                    try:
                        qc = np.nan_to_num(read_to_grid(it.assets[qk].href, grid, Resampling.nearest), nan=255).astype('int64')
                        a[((qc & 3) > 1) | (((qc >> 6) & 3) > 1)] = np.nan     # keep good/other quality, LST error <= 2 K
                    except Exception:
                        pass
                a = np.where(a > 7500, a * 0.02 - 273.15, np.nan)            # scale factor 0.02 K, 0 = fill
                acc[part].append(a)
        out[y] = {k: (np.nanmean(np.stack(v), 0) if v else None) for k, v in acc.items()}
        print(f"{y}: {len(items)} Aqua 8-day composites | day {len(acc['day'])} | night {len(acc['night'])}")
    return out, grid, centre

def rural_mask(grid, centre):
    T, W, H = grid
    xs = T.c + (np.arange(W) + 0.5) * 1000; ys = T.f - (np.arange(H) + 0.5) * 1000
    X, Y = np.meshgrid(xs, ys); dist = np.hypot(X - centre.x, Y - centre.y) / 1000
    ring = (dist >= RURAL_RING_KM[0]) & (dist <= RURAL_RING_KM[1])
    bbox = list(gpd.GeoSeries([box(*extent(grid)[0:1] + [extent(grid)[2], extent(grid)[1], extent(grid)[3]])], crs=CRS_M).to_crs(4326).total_bounds)
    note = []
    try:
        wc = get_worldcover(bbox, grid, Resampling.mode, ovr=4)
        from scipy.ndimage import binary_dilation
        built = binary_dilation(wc == 50, iterations=3)
        ring &= np.isin(wc, [20, 30, 60]) & ~built; note.append('land cover')
    except Exception as e:
        print('   WorldCover for the rural mask not read:', str(e)[:80])
    try:
        dem = get_dem(bbox, grid, Resampling.average, ovr=3)
        cx, cy = int((centre.x - T.c) / 1000), int((T.f - centre.y) / 1000)
        ring &= np.abs(dem - np.nanmedian(dem[max(cy - 10, 0):cy + 10, max(cx - 10, 0):cx + 10])) <= 150; note.append('elevation')
    except Exception as e:
        print('   DEM for the rural mask not read:', str(e)[:80])
    return ring, ' + '.join(note)

MODIS_ROWS = []
try:
    MOD, mg, mc = modis_summers()
    RURAL, rnote = rural_mask(mg, mc)
    print(f'Rural reference: {RURAL.sum()} pixels of 1 km² (filters: {rnote or "distance only"})')
    T, W, H = mg
    for name, d in D.items():
        fine = (rasterio.transform.from_origin(T.c, T.f, 100, 100), W * 10, H * 10)
        ins = inside_mask(d['poly_m'], fine)
        for y, v in MOD.items():
            row = {'district': name, 'year': y}
            for part in ('day', 'night'):
                if v[part] is None: continue
                a = np.repeat(np.repeat(v[part], 10, 0), 10, 1)
                row[f'LST_{part}'] = float(np.nanmean(a[ins])); row[f'rural_{part}'] = float(np.nanmedian(v[part][RURAL]))
                row[f'SUHI_{part}'] = row[f'LST_{part}'] - row[f'rural_{part}']
            MODIS_ROWS.append(row)
    MODIS_TABLE = pd.DataFrame(MODIS_ROWS); MODIS_TABLE.to_csv(f'{OUT}/table_modis_suhi_by_year.csv', index=False)
    display(MODIS_TABLE.groupby('district')[[c for c in MODIS_TABLE if c.startswith(('LST', 'SUHI'))]].agg(['mean', 'std']).round(2))
    for part in ('day', 'night'):
        stack = [v[part] for v in MOD.values() if v[part] is not None]
        if stack:
            mean = np.nanmean(np.stack(stack), 0); save_tif(mean, f'{OUT}/MODIS_summer_{part}_LST_mean_1km.tif', mg); globals()[f'MODIS_{part.upper()}'] = mean
    save_tif(RURAL.astype('float32'), f'{OUT}/MODIS_rural_reference_mask.tif', mg)
except Exception as e:
    print('MODIS step did not complete:', repr(e)[:300], '\n(The rest of the notebook does not depend on it. Re-run this cell later.)')

## Step 12 · What-if scenarios: cool pavements and shade sails (≈30–40 min)
1. **Cool pavement:** streets and pedestrian space resurfaced with a solar-reflective coating (albedo 0.45). The sunlit surface heats less, in proportion to the absorbed sunlight, but reflects more sunlight onto pedestrians.
2. **Shade sails:** dense shade cloth 4 m above all pedestrian space (whole lane in shared lanes, sidewalks elsewhere), letting about 3% of the sun through. The sails are treated like a tree crown at air temperature, so their own heat is not modelled (a limitation to state).
3. **Both.**

In [ ]:
from scipy.stats import wilcoxon
KEEP = ['seg', 'shade_afternoon', 'shade_day_mean', 'utci_15', 'tmrt_15', 'utci_12', 'tmrt_12', 'utci_ge46_hours', 'utci_day_mean']
DIST_H['scenario'] = 'baseline'
SCEN_H, SCEN_SEG = [DIST_H.copy()], [d['seg'][KEEP].assign(district=n, scenario='baseline') for n, d in D.items()]
if RUN_SCENARIOS:
    for name, d in D.items():
        ped, carr = d['sidewalk'], d['carriage']
        paved = (ped | carr) & d['ground']
        alb2 = np.where(paved, COOL_ALBEDO, d['albedo']).astype('float32')
        tgs = ((1 - alb2) / (1 - d['albedo'])).astype('float32')
        print(f"{name}: cool pavement on {paved.sum() * RES * RES / 1e4:.1f} ha | shade sails over {ped.sum() * RES * RES / 1e4:.1f} ha")
        for scen in ['cool pavement', 'shade sails', 'sails + cool pavement']:
            vt, vb = d['veg_top'].copy(), d['veg_base'].copy()
            if 'sails' in scen:
                sail = ped & ~np.isfinite(vt)
                vt = np.where(sail, d['dem'] + SAIL_TOP, vt).astype('float32'); vb = np.where(sail, d['dem'] + SAIL_BASE, vb).astype('float32')
                svf = sky_view_factor(d['dsm'], RES, vt, n_az=24)
            else:
                svf = d['svf']
            cool = 'cool' in scen
            seg_h, dist_h, daily_r, _ = run_day(DAY, WX, d['dsm'], vt, vb, d['ground'], svf, d['labels'], d['n_seg'], d['lat'], d['lon'],
                                                UTC_OFFSET, HOURS, RES, utci_fn=utci_fn, albedo_g=alb2 if cool else d['albedo'],
                                                key_hours=(), log=lambda *a: None, tg_scale=tgs if cool else None)
            summ = segment_summary(seg_h, daily_r, d['labels'], d['n_seg']).reset_index()
            dist_h['district'] = name; dist_h['scenario'] = scen; SCEN_H.append(dist_h)
            SCEN_SEG.append(summ[KEEP].assign(district=name, scenario=scen))
            print(f"   {scen:22s} shade 15–18 h {summ.shade_afternoon.median():5.1f}% | UTCI 12:00 {summ.utci_12.median():5.1f} | 15:00 {summ.utci_15.median():5.1f} °C | hours ≥ 46 °C {summ.utci_ge46_hours.median():4.1f}")
SCEN_H = pd.concat(SCEN_H, ignore_index=True); SCEN_SEG = pd.concat(SCEN_SEG, ignore_index=True)
SCEN_H.to_csv(f'{OUT}/scenarios_hourly.csv', index=False); SCEN_SEG.to_csv(f'{OUT}/scenarios_segments.csv', index=False)
rows = []
for (n, sc), g in SCEN_SEG.groupby(['district', 'scenario'], sort=False):
    b = SCEN_SEG[(SCEN_SEG.district == n) & (SCEN_SEG.scenario == 'baseline')].set_index('seg'); g = g.set_index('seg')
    r = {'district': n, 'scenario': sc}
    for m_ in ['shade_afternoon', 'utci_12', 'utci_15', 'tmrt_15', 'utci_ge46_hours']:
        r[m_] = round(g[m_].median(), 2)
        if sc != 'baseline':
            dd_ = (g[m_] - b[m_]).dropna(); r[f'd_{m_}'] = round(dd_.median(), 2)
            r[f'p_{m_}'] = wilcoxon(dd_).pvalue if (dd_ != 0).any() else 1.0
    rows.append(r)
SCEN_TABLE = pd.DataFrame(rows); SCEN_TABLE.to_csv(f'{OUT}/table_scenarios.csv', index=False)
display(SCEN_TABLE)

## Step 13 · Check against the official SOLWEIG model (≈15–25 min)
Runs the UMEP `solweig` package on a 400 m × 400 m window in the middle of each district at 09, 12, 15 and 18 h.
**Timing fix from paper 1:** SOLWEIG places the sun at the middle of the hour (e.g. 17:30 for the 18:00 record), so this notebook is compared at the same sun position.
If the step fails, the other results are unaffected.

In [ ]:
CHECK = []
CHECK_HALF_M, CHECK_HOURS, MARGIN_M = 200, [9, 12, 15, 18], 200
try:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'solweig'], check=True)
    import solweig
    from rasterio.windows import Window, transform as win_transform
    from datetime import datetime as _dt
    for n, d in D.items():
        sunrise, altmax = day_geometry(DAY, d['lat'], d['lon'], UTC_OFFSET)
        T, W, H = d['grid']; c = d['poly_m'].centroid
        hp, mg_ = int(CHECK_HALF_M / RES), int(MARGIN_M / RES)
        c0, r0 = int((c.x - T.c) / T.a) - hp, int((c.y - T.f) / T.e) - hp
        win = (slice(r0, r0 + 2 * hp), slice(c0, c0 + 2 * hp))
        big = (slice(r0 - mg_, r0 + 2 * hp + mg_), slice(c0 - mg_, c0 + 2 * hp + mg_))
        gw = (win_transform(Window(c0, r0, 2 * hp, 2 * hp), T), 2 * hp, 2 * hp)
        od = f"/content/solweig_{n.replace(' ', '_')}"; os.makedirs(od, exist_ok=True)
        for k, a in [('DSM', d['dsm']), ('CDSM', d['can']), ('DEM', d['dem'])]: save_tif(a[win], f'{od}/{k}_window.tif', gw)
        kw = dict(dsm=f'{od}/DSM_window.tif', cdsm=f'{od}/CDSM_window.tif', dem=f'{od}/DEM_window.tif', working_dir=f'{od}/cache')
        try: surf = solweig.SurfaceData.prepare(**kw, cdsm_relative=True)
        except TypeError: surf = solweig.SurfaceData.prepare(**kw)
        loc = solweig.Location(latitude=d['lat'], longitude=d['lon'], utc_offset=UTC_OFFSET)
        wea = []
        for h in CHECK_HOURS:
            wh = WX.loc[pd.Timestamp(_dt(DAY.year, DAY.month, DAY.day, h))]
            wea.append(solweig.Weather(datetime=_dt(DAY.year, DAY.month, DAY.day, h), ta=float(wh.ta), rh=float(wh.rh), global_rad=float(wh.ghi), ws=float(wh.ws)))
        try: solweig.calculate(surf, weather=wea, location=loc, output_dir=od, outputs=['tmrt', 'shadow'], max_shadow_distance_m=200)
        except TypeError: solweig.calculate(surf, weather=wea, location=loc, output_dir=od, outputs=['tmrt', 'shadow'])
        # this notebook's model at the same sun position (h - 30 min), on the window plus a 200 m margin for shadows
        dsm_b, vt_b, vb_b = d['dsm'][big], d['veg_top'][big], d['veg_base'][big]
        svfb, svfa = d['svf'][0][big], d['svf'][1][big]
        inner = np.zeros((2 * hp, 2 * hp), bool); m_ = int(50 / RES); inner[m_:-m_, m_:-m_] = True
        for h in CHECK_HOURS:
            f = sorted(glob.glob(f'{od}/**/tmrt*_{h:02d}00*.tif', recursive=True))
            if not f: print(f'   {n} {h}:00 - no SOLWEIG output'); continue
            with rasterio.open(f[0]) as src: off = src.read(1, masked=True).astype('float32').filled(np.nan)
            wh = WX.loc[pd.Timestamp(_dt(DAY.year, DAY.month, DAY.day, h))]
            alt, az = sun_position(_dt(DAY.year, DAY.month, DAY.day, h - 1, 30), d['lat'], d['lon'], UTC_OFFSET)
            sb, sa = shadows(dsm_b, alt, az, RES, vt_b, vb_b)
            tg = surface_temp_excess(h - 0.5, sunrise, altmax, alt, wh.ghi)
            mine = tmrt_simplified(wh.ta, wh.rh, wh.ghi, wh.dni, wh.dhi, alt, az, sa, svfa, tg, tg, svf_bldg=svfb, albedo_g=d['albedo'][big])
            mine = mine[mg_:mg_ + 2 * hp, mg_:mg_ + 2 * hp]; shade = 1 - sa[mg_:mg_ + 2 * hp, mg_:mg_ + 2 * hp]
            sw = inner & (d['labels'][win] > 0) & np.isfinite(off) & np.isfinite(mine)
            diff = mine[sw] - off[sw]
            row = {'district': n, 'hour': h, 'pixels': int(sw.sum()), 'r': round(float(np.corrcoef(mine[sw], off[sw])[0, 1]), 3),
                   'bias_C': round(float(diff.mean()), 2), 'RMSE_C': round(float(np.sqrt((diff ** 2).mean())), 2)}
            fs = sorted(glob.glob(f'{od}/**/shadow*_{h:02d}00*.tif', recursive=True))
            if fs:
                with rasterio.open(fs[0]) as src: osh = src.read(1).astype('float32')
                agree = 100 * float(np.mean((shade[sw] >= .5) == (osh[sw] < .5)))
                row['shade_agreement_pct'] = round(max(agree, 100 - agree), 1)
            row['solweig_version'] = getattr(solweig, '__version__', '?'); CHECK.append(row)
        print(f'{n}: SOLWEIG check done')
    CHECK = pd.DataFrame(CHECK); CHECK.to_csv(f'{OUT}/table_solweig_crosscheck.csv', index=False); display(CHECK)
except Exception as e:
    print('SOLWEIG check did not run:', repr(e)[:300])

## Step 14 · Statistics
- District comparison of street indicators: medians (IQR), Mann–Whitney U, **Cliff's δ** effect sizes.
- The same comparison **within each type of pedestrian space** (shared lane, street with sidewalks, pedestrian way).
- Three OLS models with standard errors clustered by 250 m block, and residual Moran's I.

In [ ]:
from scipy.stats import mannwhitneyu, spearmanr
from scipy.spatial import cKDTree
import statsmodels.formula.api as smf
SEG = pd.concat([d['seg'].assign(district=n) for n, d in D.items()], ignore_index=True)
SEG = gpd.GeoDataFrame(SEG, geometry='geometry', crs=CRS_M); SEG = SEG[SEG['ped_m2'] > 0].copy()
mid = SEG.geometry.interpolate(0.5, normalized=True); SEG['mx'], SEG['my'] = mid.x.values, mid.y.values
SEG['block'] = SEG['district'] + '_' + (SEG['mx'] // 250).astype(int).astype(str) + '_' + (SEG['my'] // 250).astype(int).astype(str)
to_gpkg(SEG, f'{OUT}/segments_results.gpkg'); SEG.drop(columns='geometry').to_csv(f'{OUT}/segments_results.csv', index=False)

METRICS = {'shade_afternoon': 'Shade 15–18 h (%)', 'shade_day_mean': 'Shade 07–18 h (%)', 'shade_bldg_day_mean': 'Shade from buildings (%)',
           'utci_12': 'UTCI 12:00 (°C)', 'utci_15': 'UTCI 15:00 (°C)', 'tmrt_15': 'Tmrt 15:00 (°C)', 'utci_ge46_hours': 'Hours UTCI ≥ 46 °C',
           'svf': 'Sky view factor', 'street_w': 'Street width (m)', 'hw_ratio': 'H/W', 'canopy_pct': 'Canopy (%)', 'lst': 'Summer LST (°C)'}
def cliffs(a, b):
    u, p = mannwhitneyu(a, b); cd = 2 * u / (len(a) * len(b)) - 1
    return p, cd, ('negligible' if abs(cd) < .147 else 'small' if abs(cd) < .33 else 'medium' if abs(cd) < .474 else 'large')
def compare(df, label):
    rows = []
    for m_, lab in METRICS.items():
        if m_ not in df: continue
        a = df.loc[df.district == 'Old core', m_].dropna(); b = df.loc[df.district == 'Al-Waha', m_].dropna()
        if len(a) < 5 or len(b) < 5: continue
        p, cd, eff = cliffs(a, b)
        rows.append({'group': label, 'indicator': lab, 'Old core': f'{a.median():.2f} ({a.quantile(.25):.2f}–{a.quantile(.75):.2f})', 'n old core': len(a),
                     'Al-Waha': f'{b.median():.2f} ({b.quantile(.25):.2f}–{b.quantile(.75):.2f})', 'n Al-Waha': len(b),
                     'p': f'{p:.3g}', "Cliff's δ": round(cd, 2), 'effect': eff})
    return rows
rows = compare(SEG, 'all streets')
for rt, g in SEG.groupby('realm'): rows += compare(g, rt)
TABLE = pd.DataFrame(rows); TABLE.to_csv(f'{OUT}/table_district_comparison.csv', index=False); display(TABLE[TABLE.group == 'all streets'])

def morans_i(x, y, z, k=8, perms=499, seed=0):
    _, idx = cKDTree(np.c_[x, y]).query(np.c_[x, y], k=k + 1); nb = idx[:, 1:]
    z = z - z.mean(); den = (z ** 2).sum(); I = (z * z[nb].mean(1)).sum() / den
    rng = np.random.default_rng(seed); sims = [(lambda q: (q * q[nb].mean(1)).sum() / den)(rng.permutation(z)) for _ in range(perms)]
    return I, (np.sum(np.abs(sims) >= abs(I)) + 1) / (perms + 1)
MODELS = {'M1 shade 15–18 h': 'shade_afternoon ~ hw_ratio + canopy_pct + C(orient) + C(realm) + C(district)',
          'M2 UTCI 15:00': 'utci_15 ~ svf + canopy_pct + C(orient) + C(realm) + C(district)',
          'M3 hours UTCI ≥ 46': 'utci_ge46_hours ~ hw_ratio + canopy_pct + C(orient) + C(realm) + C(district)'}
tabs, diag = {}, []
for title, fml in MODELS.items():
    dat = SEG[['shade_afternoon', 'utci_15', 'utci_ge46_hours', 'hw_ratio', 'svf', 'canopy_pct', 'orient', 'realm', 'district', 'block', 'mx', 'my']].dropna()
    grp = pd.factorize(dat['block'])[0]
    m_ = smf.ols(fml, data=dat).fit(cov_type='cluster', cov_kwds={'groups': grp})
    I, p = morans_i(dat.mx.values, dat.my.values, m_.resid.values)
    print(f'\n{title} | n = {int(m_.nobs)} | R² = {m_.rsquared:.3f} | {len(set(grp))} clusters | residual Moran I = {I:.3f} (p = {p:.3f})')
    print(m_.summary().tables[1]); tabs[title] = m_.summary2().tables[1]
    diag.append({'model': title, 'n': int(m_.nobs), 'R2': round(m_.rsquared, 3), 'clusters': len(set(grp)), 'moran_I': round(I, 3), 'moran_p': round(p, 3)})
pd.concat(tabs).to_csv(f'{OUT}/table_models.csv'); pd.DataFrame(diag).to_csv(f'{OUT}/table_model_diagnostics.csv', index=False)
if 'lst' in SEG and SEG['lst'].notna().any():
    for n, g in SEG.groupby('district'):
        g = g.dropna(subset=['lst', 'utci_15']); print(f"{n}: Spearman ρ(LST, UTCI 15:00) = {spearmanr(g.lst, g.utci_15).statistic:.2f} (n = {len(g)})")

## Step 15 · Figures (300 dpi, in the `figures` folder)

In [ ]:
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
COL = {'Old core': '#9e2a1f', 'Al-Waha': '#2a6f97'}

# 1 hourly shade and UTCI
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for n, g in DIST_H.groupby('district'):
    axs[0].plot(g.hour, g.sidewalk_shade_pct, '-o', ms=3, color=COL[n], label=n)
    axs[1].plot(g.hour, g.sidewalk_utci, '-o', ms=3, color=COL[n], label=n)
for lo, hi, c in [(32, 38, '#fde2c8'), (38, 46, '#f8b48a'), (46, 60, '#f08a6b')]: axs[1].axhspan(lo, hi, color=c, alpha=.5, lw=0)
axs[0].set_ylabel('Shaded pedestrian space (%)'); axs[0].set_ylim(0, 100); axs[1].set_ylabel('Mean UTCI (°C)'); axs[1].set_ylim(26, 54)
for a in axs: a.set_xlabel('Hour'); a.legend(frameon=False)
fig.tight_layout(); fig.savefig(f'{FIG}/F1_hourly_shade_utci.png', dpi=300); plt.show()

# 2 maps at 15:00
fig, axs = plt.subplots(2, 2, figsize=(11, 11))
for r, (n, d) in enumerate(D.items()):
    ex = extent(d['grid'])
    axs[r, 0].imshow(np.where(d['ground'], d['key'][15]['shade'], np.nan), extent=ex, cmap='Greys', vmin=0, vmax=1)
    im = axs[r, 1].imshow(d['key'][15]['utci'], extent=ex, cmap='inferno', vmin=40, vmax=52)
    for a in axs[r]:
        gpd.GeoSeries([d['poly_m']]).boundary.plot(ax=a, color='#00b7c7', lw=1.2); a.set_xticks([]); a.set_yticks([]); scalebar(a)
    axs[r, 0].set_title(f'{n}: shade at 15:00 (black)'); axs[r, 1].set_title(f'{n}: UTCI at 15:00'); plt.colorbar(im, ax=axs[r, 1], shrink=.7, label='°C')
fig.tight_layout(); fig.savefig(f'{FIG}/F2_maps_1500.png', dpi=300); plt.show()

# 3 by type of pedestrian space
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
for a, m_, t in zip(axs, ['shade_afternoon', 'utci_15', 'utci_ge46_hours'], ['Shade 15–18 h (%)', 'UTCI 15:00 (°C)', 'Hours UTCI ≥ 46 °C']):
    cats = [c for c in ['shared lane', 'pedestrian way', 'street with sidewalks'] if c in set(SEG.realm)]
    for i, n in enumerate(D):
        data = [SEG[(SEG.district == n) & (SEG.realm == c)][m_].dropna() for c in cats]
        pos = np.arange(len(cats)) + (i - .5) * .35
        bp = a.boxplot([x if len(x) else [np.nan] for x in data], positions=pos, widths=.3, patch_artist=True, showfliers=False)
        for b_ in bp['boxes']: b_.set_facecolor(COL[n]); b_.set_alpha(.6)
    a.set_xticks(range(len(cats))); a.set_xticklabels(cats, fontsize=8); a.set_title(t)
fig.tight_layout(); fig.savefig(f'{FIG}/F3_by_pedestrian_space.png', dpi=300); plt.show()

# 4 LCZ maps
pal = {'LCZ 1': '#8c0000', 'LCZ 2': '#d10000', 'LCZ 3': '#ff0000', 'LCZ 4': '#bf4d00', 'LCZ 5': '#ff6600', 'LCZ 6': '#ff9955',
       'LCZ 8': '#bcbcbc', 'LCZ 9': '#ffccaa', 'LCZ E': '#fbf7ae', 'LCZ F': '#f4ddb3'}
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
for a, (n, d) in zip(axs, D.items()):
    c = d['cells']; cl = c.lcz.str[:5]
    gpd.GeoDataFrame(c, geometry=[box(x - 50, y - 50, x + 50, y + 50) for x, y in zip(c.x, c.y)], crs=CRS_M).plot(ax=a, color=[pal.get(k, '#999') for k in cl], edgecolor='white', lw=.2)
    gpd.GeoSeries([d['poly_m']]).boundary.plot(ax=a, color='k', lw=1); a.set_axis_off(); a.set_title(n)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=v, label=k) for k, v in pal.items() if k in set(LCZ.lcz.str[:5])], loc='lower center', ncol=6, frameon=False)
fig.savefig(f'{FIG}/F4_lcz.png', dpi=300, bbox_inches='tight'); plt.show()

# 5 LST and hot spots
if all(d.get('lst30') is not None for d in D.values()):
    fig, axs = plt.subplots(2, 2, figsize=(11, 10))
    vmin = min(np.nanpercentile(d['lst30'], 2) for d in D.values()); vmax = max(np.nanpercentile(d['lst30'], 98) for d in D.values())
    from matplotlib.colors import ListedColormap, BoundaryNorm
    gcm = ListedColormap(['#2166ac', '#67a9cf', '#d1e5f0', '#f7f7f7', '#fddbc7', '#ef8a62', '#b2182b']); gn = BoundaryNorm(np.arange(-3.5, 4.5), 7)
    for r, (n, d) in enumerate(D.items()):
        ex = extent(d['g30'])
        im = axs[r, 0].imshow(d['lst30'], extent=ex, cmap='magma', vmin=vmin, vmax=vmax)
        im2 = axs[r, 1].imshow(d['gi'], extent=ex, cmap=gcm, norm=gn)
        for a in axs[r]:
            gpd.GeoSeries([d['poly_m']]).boundary.plot(ax=a, color='#00b7c7' if a is axs[r, 0] else 'k', lw=1.2); a.set_xticks([]); a.set_yticks([]); scalebar(a, 1000)
        axs[r, 0].set_title(f'{n}: summer LST'); axs[r, 1].set_title(f'{n}: hot and cold spots (Gi*)')
        plt.colorbar(im, ax=axs[r, 0], shrink=.7, label='°C')
        cb = plt.colorbar(im2, ax=axs[r, 1], shrink=.7, ticks=range(-3, 4)); cb.ax.set_yticklabels(['cold 99%', '95%', '90%', 'n.s.', '90%', '95%', 'hot 99%'])
    fig.tight_layout(); fig.savefig(f'{FIG}/F5_lst_hotspots.png', dpi=300); plt.show()

# 6 MODIS day/night
if 'MODIS_TABLE' in globals() and len(MODIS_TABLE):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
    ex = extent(mg)
    for a, part in zip(axs[:2], ['day', 'night']):
        if f'MODIS_{part.upper()}' in globals():
            im = a.imshow(globals()[f'MODIS_{part.upper()}'], extent=ex, cmap='inferno'); plt.colorbar(im, ax=a, shrink=.7, label='°C')
            a.contour(np.flipud(RURAL.astype(float)), levels=[.5], colors='cyan', linewidths=.6, extent=ex, origin='lower')
            for n, d in D.items(): gpd.GeoSeries([d['poly_m']]).boundary.plot(ax=a, color='white', lw=1.2)
            a.set_title(f'Summer LST, {"13:30" if part == "day" else "01:30"} ({MODIS_YEARS[0]}–{MODIS_YEARS[1]})'); a.set_xticks([]); a.set_yticks([])
    for n, g in MODIS_TABLE.groupby('district'):
        if 'SUHI_day' in g: axs[2].plot(g.year, g.SUHI_day, '-o', ms=3, color=COL[n], label=f'{n}, day')
        if 'SUHI_night' in g: axs[2].plot(g.year, g.SUHI_night, '--s', ms=3, color=COL[n], label=f'{n}, night')
    axs[2].axhline(0, color='k', lw=.6); axs[2].set_ylabel('SUHI vs desert (°C)'); axs[2].legend(frameon=False, fontsize=8); axs[2].set_title('Surface heat-island intensity')
    fig.tight_layout(); fig.savefig(f'{FIG}/F6_modis_day_night.png', dpi=300); plt.show()

# 7 scenarios
if SCEN_H['scenario'].nunique() > 1:
    SC = {'baseline': '#4d4d4d', 'cool pavement': '#e09f3e', 'shade sails': '#335c67', 'sails + cool pavement': '#9e2a2b'}
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    for (n, sc), g in SCEN_H.groupby(['district', 'scenario'], sort=False):
        ls = '-' if n == 'Old core' else '--'
        axs[0].plot(g.hour, g.sidewalk_tmrt, ls, color=SC[sc], label=sc if n == 'Old core' else None)
        axs[1].plot(g.hour, g.sidewalk_utci, ls, color=SC[sc])
    axs[0].set_ylabel('Mean Tmrt (°C)'); axs[1].set_ylabel('Mean UTCI (°C)'); axs[0].legend(frameon=False, fontsize=8)
    for a in axs: a.set_xlabel('Hour (solid: old core, dashed: Al-Waha)')
    fig.tight_layout(); fig.savefig(f'{FIG}/F7_scenarios.png', dpi=300); plt.show()

## Step 16 (optional) · Link to your GIS walkability run
If you have `oldcore_ks_out.zip` from the GIS walkability notebook, upload it with the 📁 icon on the left (to `/content`) and run this cell. It finds the hexagon grid in it, adds the thermal results to every hexagon (UTCI, hours of extreme heat, shade, LST, main LCZ) and correlates them with the walkability indices. Skip it if you do not have the file.

In [ ]:
GIS_ZIP = '/content/oldcore_ks_out.zip'
if os.path.exists(GIS_ZIP):
    import pyogrio
    zdir = '/content/gis_run'; zipfile.ZipFile(GIS_ZIP).extractall(zdir)
    best = None
    for f in glob.glob(f'{zdir}/**/*.gpkg', recursive=True) + glob.glob(f'{zdir}/**/*.geojson', recursive=True):
        try: layers = [l[0] for l in pyogrio.list_layers(f)]
        except Exception: layers = [None]
        for l in layers:
            try: g = gpd.read_file(f, layer=l) if l else gpd.read_file(f)
            except Exception: continue
            if len(g) < 15 or not g.geom_type.isin(['Polygon', 'MultiPolygon']).all(): continue
            a = g.to_crs(CRS_M).area.median()
            score = ('hex' in f'{f}{l}'.lower()) * 2 + (20000 < a < 120000)
            if best is None or score > best[0]: best = (score, f, l, len(g))
    if best is None or best[0] == 0:
        print('No hexagon grid found in the zip.')
    else:
        print('hexagon grid:', best[1], best[2], f'({best[3]} hexagons)')
        HEX = gpd.read_file(best[1], layer=best[2]).to_crs(CRS_M).reset_index(drop=True); HEX['hex_id'] = np.arange(len(HEX))
        rows = []
        for n, d in D.items():
            T, W, H = d['grid']
            lab = rasterize(((gm, i + 1) for gm, i in zip(HEX.geometry, HEX.hex_id)), out_shape=(H, W), transform=T, fill=0, dtype='int32')
            lab[~(d['ground'] & d['inside'])] = 0; nh = len(HEX)
            vals = {'utci_15': d['key'][15]['utci'], 'utci_12': d['key'][12]['utci'], 'shade_15_pct': 100 * d['key'][15]['shade'],
                    'utci_ge46_hours': d['daily']['utci_ge46_hours'], 'tmrt_mean': d['daily']['tmrt_mean']}
            r = pd.DataFrame({k: zonal_mean(lab, v, nh)[1:] for k, v in vals.items()}); r['hex_id'] = HEX.hex_id; r['open_px'] = np.bincount(lab.ravel(), minlength=nh + 1)[1:]
            r = r[r.open_px > 500]; r['thermal_district'] = n; rows.append(r)
        TH = pd.concat(rows); HEXT = HEX.merge(TH, on='hex_id', how='inner')
        to_gpkg(HEXT, f'{OUT}/hexagons_walkability_thermal.gpkg'); HEXT.drop(columns='geometry').to_csv(f'{OUT}/hexagons_walkability_thermal.csv', index=False)
        wcols = [c for c in HEXT.columns if any(k in c.lower() for k in ('wi', 'walk', 'kswi', 'uf'))
                 and pd.api.types.is_numeric_dtype(HEXT[c]) and c not in ('hex_id',)]
        out = []
        for c in wcols:
            for t in ['utci_15', 'utci_ge46_hours', 'shade_15_pct']:
                ok = HEXT[[c, t]].dropna()
                if len(ok) > 10: out.append({'walkability_metric': c, 'thermal_metric': t, 'rho': spearmanr(ok[c], ok[t]).statistic, 'n': len(ok)})
        HEXCOR = pd.DataFrame(out); HEXCOR.to_csv(f'{OUT}/table_hex_walkability_vs_heat.csv', index=False)
        print(f'{len(HEXT)} hexagons linked'); display(HEXCOR.sort_values('rho').round(2).head(30))
else:
    print('No GIS walkability zip uploaded - step skipped.')

## Step 17 · Methods summary with your numbers, and save everything

In [ ]:
lines = [f'Design day: {DAY} (hottest clear-sky day, Jul–Aug {WEATHER_YEAR}; ERA5 via Open-Meteo, {season.attrs["radiation"]} radiation).']
for n, d in D.items():
    b = d['bldg']; s = d['seg']
    lines.append(f"{n}: {d['poly_m'].area/1e6:.2f} km²; {len(b):,} buildings (height sources: {b['height_src'].value_counts(normalize=True).mul(100).round(0).to_dict()}); "
                 f"{d['n_seg']} street segments, {s['length_m'].sum()/1000:.1f} km, pedestrian space {s['ped_m2'].sum()/1e4:.1f} ha; "
                 f"realm types: {s['realm'].value_counts().to_dict()}.")
if isinstance(CHECK, pd.DataFrame) and len(CHECK):
    lines.append(f"SOLWEIG check (v{CHECK.solweig_version.iloc[0]}): r {CHECK.r.min():.2f}–{CHECK.r.max():.2f}, bias {CHECK.bias_C.min():+.1f} to {CHECK.bias_C.max():+.1f} °C, "
                 f"RMSE {CHECK.RMSE_C.min():.1f}–{CHECK.RMSE_C.max():.1f} °C, shade agreement {CHECK.get('shade_agreement_pct', pd.Series([np.nan])).min():.1f}–{CHECK.get('shade_agreement_pct', pd.Series([np.nan])).max():.1f}%.")
if 'LST_TABLE' in globals() and len(LST_TABLE):
    lines.append('Landsat: ' + '; '.join(f"{r.district} {r.scenes} scenes" for r in LST_TABLE.itertuples()))
if 'MODIS_TABLE' in globals() and len(MODIS_TABLE):
    lines.append(f'MODIS Aqua MYD11A2 summers {MODIS_YEARS[0]}–{MODIS_YEARS[1]}; rural reference {int(RURAL.sum())} pixels ({rnote}).')
txt = '\n'.join(lines); print(txt); open(f'{OUT}/methods_numbers.txt', 'w').write(txt)

zpath = '/content/oldcore_alwaha_thermal_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            if not f.endswith(('.parquet', '.src')) and 'cache' not in root:
                z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print(f'{zpath}: {os.path.getsize(zpath)/1e6:.0f} MB')
if OUT.startswith('/content/drive'):
    subprocess.run(['cp', zpath, os.path.dirname(OUT) + '/'], check=True)
    print('Also saved as Google Drive > MyDrive > oldcore_alwaha_thermal_results.zip')
try:
    from google.colab import files as colab_files
    colab_files.download(zpath)
except Exception as e:
    print('Browser download did not start; take the zip from Google Drive instead.')

## If something fails
| Message | Fix |
|---|---|
| Session disconnected | **Runtime → Run all** again: finished parts are reused from Google Drive |
| `overturemaps` / `duckdb` / S3 error | Re-run the cell; if it persists, put the newest release from docs.overturemaps.org in `OVERTURE_RELEASE` |
| All OpenStreetMap servers failed | The notebook switches to Overture streets automatically |
| Planetary Computer HTTP 403 | Signed links expired; re-run that step |
| Open-Meteo 429 | Wait a minute and re-run Step 4 |
| `MemoryError` | Runtime → Change runtime type → High-RAM, or set `RES = 2.0` for a test |